In [ ]:
# =========================================
# Phase 4 - Activity Detection (Onset / Offset) - Full version with EMG fallback
# Last update: 2026-02-09 (improved for weak ALS signals and better merging)
# =========================================
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ─── Paths ────────────────────────────────────────
NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed"
PHASE2_DIR = PROCESSED_ROOT / "phase_02_preprocess_emg"
PHASE2B_DIR = PROCESSED_ROOT / "phase_02b_preprocess_imu"
PHASE3_DIR = PROCESSED_ROOT / "phase_03_normalization"
PHASE3_NORM_DIR = PHASE3_DIR / "normalized_env_per_subject"  # fixed: exact path for env_norm
PHASE4_DIR = PROCESSED_ROOT / "phase_04_events"
PHASE4_DIR.mkdir(parents=True, exist_ok=True)
print("Paths and initial settings loaded.")

# ─── Task-specific settings ─────────────────────────
TASK_SETTINGS = {
    "lifting": {"min_duration_s": 1.8, "merge_gap_s": 0.50, "min_peak_gate": 100.0},
    "drinking": {"min_duration_s": 1.2, "merge_gap_s": 1.00, "min_peak_gate": 60.0},   # increased merge_gap
    "pick&place": {"min_duration_s": 2.2, "merge_gap_s": 1.50, "min_peak_gate": 75.0},
    "default": {"min_duration_s": 1.5, "merge_gap_s": 0.60, "min_peak_gate": 70.0}
}

# Settings for EMG fallback mode (normalized envelope)
EMG_FALLBACK_SETTINGS = {
    "k_on": 1.8,               # reduced for better ALS burst detection
    "k_off": 0.9,
    "peak_multiplier": 1.1,    # accept smaller peaks
    "min_duration_factor": 0.6,
    "merge_gap_factor": 2.5,   # increased to avoid fragmentation
    "secondary_merge_gap": 4.0 # stronger merging
}

# Minimum episode duration to keep (in seconds) - shorter episodes will be discarded as non-realistic
MIN_VALID_EPISODE_DURATION = 2.0  # reduced from 3.5 for ALS
MIN_VALID_PEAK_GATE = 40.0        # reduced from 50 for weaker bursts

# ─── Episode detection with hysteresis and post-processing ────────
def detect_episodes_hysteresis(
    t: np.ndarray,
    gate: np.ndarray,
    method: str = "mad",
    k_on: float = 5.0,
    k_off: float = 3.5,
    min_duration_s: float = 1.8,
    merge_gap_s: float = 0.45,
    pad_s: float = 0.15,
    peak_multiplier: float = 2.5,
    secondary_merge_gap: float = 0.8,
) -> tuple[pd.DataFrame, dict]:
    t = np.asarray(t, dtype=float)
    gate = np.asarray(gate, dtype=float)
    mask_valid = np.isfinite(t) & np.isfinite(gate)
    t = t[mask_valid]
    gate = gate[mask_valid]
    
    log = {
        "raw_samples": len(t),
        "gate_median": float(np.median(gate)) if len(gate) > 0 else np.nan,
        "gate_95p": float(np.percentile(gate, 95)) if len(gate) > 0 else np.nan,
    }
    
    if len(t) < 20:
        log["warning"] = "too_few_samples"
        return pd.DataFrame(columns=["t_on","t_off","duration_s","thr_on","thr_off","peak_gate"]), log
    
    med = float(np.median(gate))
    mad = float(np.median(np.abs(gate - med))) + 1e-12
    thr_on = med + k_on * mad
    thr_off = med + k_off * mad
    log["thr_on"] = float(thr_on)
    log["thr_off"] = float(thr_off)
    
    # Adaptive adjustment for weak ALS signals
    gate_95p = log["gate_95p"]
    if thr_on > gate_95p * 0.85 or gate_95p < 60.0:  # stronger condition for weak signals
        k_on_new = max(1.5, k_on * 0.55)  # stronger reduction
        thr_on = med + k_on_new * mad
        k_off_new = max(0.8, k_off * 0.6)
        thr_off = med + k_off_new * mad
        log["thr_on_adjusted"] = float(thr_on)
        log["thr_off_adjusted"] = float(thr_off)
        log["warning"] = "thr_adjusted_for_weak_ALS_signal"
    
    episodes = []
    in_episode = False
    t_start = None
    
    for i in range(len(t)):
        if not in_episode:
            if gate[i] >= thr_on:
                in_episode = True
                t_start = float(t[i])
        else:
            if gate[i] <= thr_off:
                episodes.append([t_start, float(t[i])])
                in_episode = False
                t_start = None
    
    if in_episode:
        episodes.append([t_start, float(t[-1])])
        log["last_episode_open"] = True
    
    if not episodes:
        log["warning"] = "no_episodes_detected"
        return pd.DataFrame(columns=["t_on","t_off","duration_s","thr_on","thr_off","peak_gate"]), log
    
    t_min, t_max = float(t[0]), float(t[-1])
    padded = [[max(t_min, a - pad_s), min(t_max, b + pad_s)] for a, b in episodes]
    padded.sort(key=lambda x: x[0])
    
    # Merging stage 1
    merged = [padded[0]]
    for curr in padded[1:]:
        if curr[0] - merged[-1][1] <= merge_gap_s:
            merged[-1][1] = max(merged[-1][1], curr[1])
        else:
            merged.append(curr)
    
    # Merging stage 2 (stronger)
    final_merged = []
    current = merged[0]
    for nxt in merged[1:]:
        if nxt[0] - current[1] <= secondary_merge_gap:
            current[1] = max(current[1], nxt[1])
        else:
            final_merged.append(current)
            current = nxt
    final_merged.append(current)
    
    # Stronger merging for close intervals (< 1.2 s gap)
    secondary_merged = []
    current = final_merged[0] if final_merged else None
    for nxt in final_merged[1:]:
        if nxt[0] - current[1] <= 1.2:
            current[1] = max(current[1], nxt[1])
        else:
            secondary_merged.append(current)
            current = nxt
    if current:
        secondary_merged.append(current)
    final_merged = secondary_merged
    
    adaptive_peak_thr = med + peak_multiplier * mad
    log["adaptive_peak_thr"] = adaptive_peak_thr
    
    final_ep = []
    for a, b in final_merged:
        dur = b - a
        if dur < min_duration_s:
            continue
        segment_gate = gate[(t >= a) & (t <= b)]
        if len(segment_gate) == 0:
            continue
        peak = float(np.max(segment_gate))
        if peak >= adaptive_peak_thr:
            final_ep.append([a, b, dur, thr_on, thr_off, peak])
    
    df_ep = pd.DataFrame(final_ep, columns=["t_on", "t_off", "duration_s", "thr_on", "thr_off", "peak_gate"])
    log["n_episodes_after_postproc"] = len(df_ep)
    
    return df_ep, log

# ─── Build gate from IMU ────────────────────────────────────────
def build_gate(
    df_imu_trial: pd.DataFrame,
    use_acc_gate: bool = True,
    alpha: float = 1.0,
    time_bin_res: float = 0.001,
) -> tuple:
    df = df_imu_trial.copy()
    df["t_bin"] = (df["t_imu"] / time_bin_res).round().astype(int)
    agg_dict = {"t_imu": "mean", "gyro_gate": "median"}
    if use_acc_gate and "acc_gate" in df.columns:
        agg_dict["acc_gate"] = "median"
    df_agg = df.groupby("t_bin", as_index=False).agg(agg_dict).sort_values("t_imu")
    t = df_agg["t_imu"].to_numpy(dtype=float)
    gyro = df_agg["gyro_gate"].to_numpy(dtype=float)
    if use_acc_gate and "acc_gate" in df_agg:
        acc = df_agg["acc_gate"].to_numpy(dtype=float)
        gate = np.maximum(gyro, alpha * acc)
        source = f"max(gyro_gate, {alpha:.2f} × acc_gate)"
    else:
        gate = gyro
        source = "gyro_gate"
    log_info = {
        "n_sensors_used": int(df["sensor"].nunique()) if "sensor" in df else 1,
        "gate_median": float(np.median(gate)),
        "gate_95p": float(np.percentile(gate, 95)),
    }
    return t, gate, source, "median_over_sensors", log_info

# ─── Build gate from EMG (fallback) ────────────────────────────────
def build_gate_from_emg(
    df_emg_trial: pd.DataFrame,
    time_bin_res: float = 0.001,
) -> tuple:
    df = df_emg_trial.copy()
    env_cols = [col for col in df.columns if col.startswith('env_norm_')]
    if not env_cols:
        env_cols = [col for col in df.columns if 'env_norm' in col.lower()]
    if not env_cols:
        raise ValueError("No env_norm columns found. Check column names.")
    print(f" env_norm columns found for fallback: {env_cols}")
    # Important fix: use max instead of mean → more sensitive to strongest muscle activity
    df["emg_gate"] = df[env_cols].max(axis=1)
    df["t_bin"] = (df["t_emg"] / time_bin_res).round().astype(int)
    df_agg = df.groupby("t_bin", as_index=False).agg(
        {"t_emg": "mean", "emg_gate": "median"}
    ).sort_values("t_emg")
    t = df_agg["t_emg"].to_numpy(dtype=float)
    gate = df_agg["emg_gate"].to_numpy(dtype=float)
    # If variance is very low, boost gate
    gate_var = np.var(gate)
    if gate_var < 1e-5:
        gate = gate * 5.0  # adjust factor as needed
        print(" Warning: gate variance too low – scaled gate")
    source = f"max_of_{len(env_cols)}_env_norm_muscles"
    log_info = {
        "n_muscles_used": len(env_cols),
        "gate_median": float(np.median(gate)),
        "gate_95p": float(np.percentile(gate, 95)),
        "gate_var": gate_var,
    }
    return t, gate, source, "median_over_time_bins", log_info

# ─── Plot gate and episodes ──────────────────────────────────────
def plot_trial_with_episodes(subject_name, trial_id, episodes_df):
    imu_path = PHASE2B_DIR / f"{subject_name}__imu_gate.parquet"
    emg_path = PHASE3_NORM_DIR / f"{subject_name}__emg_env_norm.parquet"  # fixed: correct path
    ep = episodes_df[
        (episodes_df["subject"] == subject_name) &
        (episodes_df["trial_id"] == trial_id) &
        (episodes_df["duration_s"] >= MIN_VALID_EPISODE_DURATION) &
        (episodes_df["peak_gate"] >= MIN_VALID_PEAK_GATE)  # show valid episodes only
    ].copy().sort_values("t_on")
    plt.figure(figsize=(14, 5))
    gate_source = "Unknown"
    if imu_path.exists():
        df_imu = pd.read_parquet(imu_path)
        df_trial = df_imu[(df_imu["subject"] == subject_name) & (df_imu["trial_id"] == trial_id)]
        if not df_trial.empty:
            df_trial = df_trial.sort_values("t_imu")
            t, gate, gate_source, _, _ = build_gate(df_trial, use_acc_gate=True, alpha=1.0)
            plt.plot(t, gate, lw=1.3, label=f"gate: {gate_source} (IMU)")
    if gate_source == "Unknown" and emg_path.exists():
        df_emg = pd.read_parquet(emg_path)
        df_trial = df_emg[(df_emg["subject"] == subject_name) & (df_emg["trial_id"] == trial_id)]
        if not df_trial.empty:
            df_trial = df_trial.sort_values("t_emg")
            t, gate, gate_source, _, _ = build_gate_from_emg(df_trial)
            plt.plot(t, gate, lw=1.3, label=f"gate: {gate_source} (EMG fallback)")
    if gate_source == "Unknown":
        print(f"No valid gate built for {trial_id} (neither IMU nor EMG)")
        plt.close()
        return
    for i, (_, r) in enumerate(ep.iterrows()):
        plt.axvspan(r["t_on"], r["t_off"], alpha=0.18, color='orange')
        plt.axvline(r["t_on"], ls="--", lw=1.1, color='darkgreen', label='on' if i==0 else None)
        plt.axvline(r["t_off"], ls="--", lw=1.1, color='darkred', label='off' if i==0 else None)
    plt.title(f"{subject_name} — {trial_id} — Episodes: {len(ep)}")
    plt.xlabel("Time (seconds)")
    plt.ylabel("Gate")
    plt.grid(True, alpha=0.35)
    plt.legend()
    plt.tight_layout()
    plt.show()

# ─── Main run for one subject ────────────────────────────────
def run_phase4_for_subject(
    subject_name: str,
    method: str = "mad",
    k_on: float = 5.0,
    k_off: float = 3.50,
    pad_s: float = 0.15,
    use_acc_gate: bool = True,
    alpha_acc: float = 1.0,
    compute_sanity_check: bool = True,
):
    imu_path = PHASE2B_DIR / f"{subject_name}__imu_gate.parquet"
    emg_path = PHASE3_NORM_DIR / f"{subject_name}__emg_env_norm.parquet"  # fixed: correct path
    # Load data
    df_imu = pd.DataFrame()
    df_emg = pd.DataFrame()
    if imu_path.exists():
        df_imu = pd.read_parquet(imu_path)
        df_imu["subject"] = df_imu["subject"].astype(str)
        df_imu["trial_id"] = df_imu["trial_id"].astype(str)
    if emg_path.exists():
        df_emg = pd.read_parquet(emg_path)
        df_emg["subject"] = df_emg["subject"].astype(str)
        df_emg["trial_id"] = df_emg["trial_id"].astype(str)
    if df_imu.empty and df_emg.empty:
        raise FileNotFoundError(f"No data found for {subject_name} (neither IMU nor EMG)")
    # Union of all trials
    trials_imu = set(df_imu["trial_id"].unique()) if not df_imu.empty else set()
    trials_emg = set(df_emg["trial_id"].unique()) if not df_emg.empty else set()
    all_trials = sorted(trials_imu | trials_emg)
    print(f"Total unique trials: {len(all_trials)}")
    print(f" • With IMU: {len(trials_imu)}")
    print(f" • EMG only: {len(trials_emg - trials_imu)}")
    events = []
    logs = []
    for trial_id in all_trials:
        use_imu = trial_id in trials_imu
        if use_imu:
            df_trial = df_imu[df_imu["trial_id"] == trial_id].copy()
            source_type = "IMU"
        else:
            df_trial = df_emg[df_emg["trial_id"] == trial_id].copy()
            source_type = "EMG_fallback"
        if df_trial.empty:
            print(f"Warning: trial {trial_id} is empty → skipped")
            continue
        print(f"Processing {trial_id:30} → source: {source_type}")
        
        # ─── parameter setting by task and exo/noexo ────────
        task_key = next((k for k in TASK_SETTINGS if k in trial_id.lower()), "default")
        params = TASK_SETTINGS[task_key].copy()
        
        curr_merge_gap = params["merge_gap_s"]
        curr_sec_merge = 0.8
        curr_min_dur = params["min_duration_s"]
        curr_peak_mult = 2.5
        
        # stronger merging for repetitive tasks with pauses (drinking, lifting, pick&place)
        if task_key in ["drinking", "lifting", "pick&place"]:
            curr_merge_gap = max(curr_merge_gap, 1.2)
            curr_sec_merge = max(curr_sec_merge, 3.5)
            print(f" → Increased merging for task '{task_key}': merge_gap={curr_merge_gap}s, secondary={curr_sec_merge}s")
        
        # special setting for noexo (usually more fragmented in ALS)
        if "noexo" in trial_id.lower():
            curr_merge_gap *= 1.6
            curr_sec_merge = max(curr_sec_merge, 4.0)
            curr_min_dur *= 0.75   # allow slightly shorter episodes
            print(f" → Extra merging & duration relax for noexo: merge_gap={curr_merge_gap}s")
        
        # ─── Gate quality check and source selection ────────
        if use_imu:
            t, gate, gate_source, _, gate_log = build_gate(
                df_trial, use_acc_gate=use_acc_gate, alpha=alpha_acc
            )
            curr_k_on = k_on
            curr_k_off = k_off
            gate_log["data_source"] = "imu"
        else:
            t, gate, gate_source, _, gate_log = build_gate_from_emg(df_trial)
            curr_k_on = EMG_FALLBACK_SETTINGS["k_on"]
            curr_k_off = EMG_FALLBACK_SETTINGS["k_off"]
            curr_peak_mult = EMG_FALLBACK_SETTINGS["peak_multiplier"]
            curr_min_dur *= EMG_FALLBACK_SETTINGS["min_duration_factor"]
            curr_merge_gap *= EMG_FALLBACK_SETTINGS["merge_gap_factor"]
            curr_sec_merge = EMG_FALLBACK_SETTINGS["secondary_merge_gap"]
            gate_log["data_source"] = "emg_only"
        
        # Gate variance check
        if compute_sanity_check:
            gate_var = np.var(gate)
            gate_log["gate_var"] = gate_var
            if gate_var < 1e-4:
                gate = gate * 1.5  # small boost for very flat signals
                print(f" → Gate boosted ×1.5 due to very low variance ({gate_var:.6f})")
                gate_log["sanity_flag"] = "low_variance_boosted"
        
        # ─── episode detection ────────
        ep_df, ep_log = detect_episodes_hysteresis(
            t, gate,
            method=method,
            k_on=curr_k_on,
            k_off=curr_k_off,
            min_duration_s=curr_min_dur,
            merge_gap_s=curr_merge_gap,
            pad_s=pad_s,
            peak_multiplier=curr_peak_mult,
            secondary_merge_gap=curr_sec_merge,
        )
        
        # ─── Post-processing final filter (new values) ────────
        if not ep_df.empty:
            before_count = len(ep_df)
            ep_df = ep_df[ep_df["duration_s"] >= MIN_VALID_EPISODE_DURATION].copy()
            ep_df = ep_df[ep_df["peak_gate"] >= MIN_VALID_PEAK_GATE].copy()
            discarded = before_count - len(ep_df)
            if discarded > 0:
                print(f" → Discarded {discarded} episodes (short or low peak) in {trial_id}")
                ep_log["discarded_episodes"] = discarded
                ep_log["min_valid_duration"] = MIN_VALID_EPISODE_DURATION
                ep_log["min_valid_peak_gate"] = MIN_VALID_PEAK_GATE
        
        log = {
            "subject": subject_name,
            "trial_id": trial_id,
            "task_type": task_key,
            "gate_source": gate_source,
            "data_source": source_type,
            "method": method,
            **gate_log,
            **ep_log,
        }
        if ep_df.empty:
            print(f" → Warning: no episodes detected for {trial_id} – log: {ep_log.get('warning', 'unknown')}")
            logs.append(log)
            continue
        
        # Calculate peak_gate for remaining episodes
        ep_df["peak_gate"] = [
            float(np.nanmax(gate[(t >= a) & (t <= b)])) if np.any((t >= a) & (t <= b)) else np.nan
            for a, b in zip(ep_df["t_on"], ep_df["t_off"])
        ]
        ep_df = ep_df.sort_values("t_on").reset_index(drop=True)
        ep_df["episode_id"] = range(1, len(ep_df) + 1)
        
        for _, row in ep_df.iterrows():
            r = {
                "subject": subject_name,
                "trial_id": trial_id,
                "episode_id": int(row["episode_id"]),
                "t_on": float(row["t_on"]),
                "t_off": float(row["t_off"]),
                "duration_s": float(row["duration_s"]),
                "peak_gate": row["peak_gate"],
                "thr_on": float(row["thr_on"]),
                "thr_off": float(row["thr_off"]),
                "gate_source": gate_source,
                "task_type": task_key,
                "data_source": source_type,
            }
            events.append(r)
        logs.append(log)
    
    events_df = pd.DataFrame(events)
    logs_df = pd.DataFrame(logs)
    # Save
    events_df.to_parquet(PHASE4_DIR / f"{subject_name}__episodes.parquet", index=False)
    logs_df.to_csv(PHASE4_DIR / f"{subject_name}__episodes_log.csv", index=False)
    print(f"\nProcessing {subject_name} completed.")
    print(f" → episodes: {PHASE4_DIR / f'{subject_name}__episodes.parquet'}")
    print(f" → log: {PHASE4_DIR / f'{subject_name}__episodes_log.csv'}")
    return events_df, logs_df

In [ ]:
# ─── Run cell ────────────────────────────────────────────────
SUBJECT_NAME = "ALS_Subject_8"   # <- change only this line

print(f"\n=== Starting Phase 4 processing for {SUBJECT_NAME} ===\n")
print(f"Current filter settings: min_duration={MIN_VALID_EPISODE_DURATION}s | min_peak_gate={MIN_VALID_PEAK_GATE}")

events_df, logs_df = run_phase4_for_subject(
    subject_name = SUBJECT_NAME,
    method       = "mad",
    k_on         = 5.0,
    k_off        = 3.50,
    pad_s        = 0.15,
    use_acc_gate = True,
    alpha_acc    = 1.0,
    compute_sanity_check = True,
)

# ─── Display summary results ────────────────────────────────
if not events_df.empty:
    print("\nEpisode duration distribution (seconds):")
    display(events_df["duration_s"].describe())
    
    print("\nNumber of episodes per trial and data source:")
    display(events_df.groupby(["trial_id", "data_source"])["episode_id"].count().unstack(fill_value=0))
    
    # new: summary of discarded episodes from logs_df (if the filter fired)
    if 'discarded_episodes' in logs_df.columns:
        total_discarded = logs_df["discarded_episodes"].sum()
        print(f"\nTotal discarded short/weak episodes across all trials: {total_discarded}")
        display(logs_df[logs_df["discarded_episodes"] > 0][["trial_id", "discarded_episodes", "task_type"]])
    
    print(f"\nPlotting {len(events_df['trial_id'].unique())} valid trials (after filtering):")
    for tid in sorted(events_df["trial_id"].unique()):
        print(f"  → {tid}")
        plot_trial_with_episodes(SUBJECT_NAME, tid, events_df)
else:
    print("No episodes detected after filtering → check logs.")
    display(logs_df)

In [ ]:
# ─── Run cell ────────────────────────────────────────────────
SUBJECT_NAME = "ALS_Subject_9"   # <- change only this line

print(f"\n=== Starting Phase 4 processing for {SUBJECT_NAME} ===\n")
print(f"Current filter settings: min_duration={MIN_VALID_EPISODE_DURATION}s | min_peak_gate={MIN_VALID_PEAK_GATE}")

events_df, logs_df = run_phase4_for_subject(
    subject_name = SUBJECT_NAME,
    method       = "mad",
    k_on         = 5.0,
    k_off        = 3.50,
    pad_s        = 0.15,
    use_acc_gate = True,
    alpha_acc    = 1.0,
    compute_sanity_check = True,
)

# ─── Display summary results ────────────────────────────────
if not events_df.empty:
    print("\nEpisode duration distribution (seconds):")
    display(events_df["duration_s"].describe())
    
    print("\nNumber of episodes per trial and data source:")
    display(events_df.groupby(["trial_id", "data_source"])["episode_id"].count().unstack(fill_value=0))
    
    # new: summary of discarded episodes from logs_df (if the filter fired)
    if 'discarded_episodes' in logs_df.columns:
        total_discarded = logs_df["discarded_episodes"].sum()
        print(f"\nTotal discarded short/weak episodes across all trials: {total_discarded}")
        display(logs_df[logs_df["discarded_episodes"] > 0][["trial_id", "discarded_episodes", "task_type"]])
    
    print(f"\nPlotting {len(events_df['trial_id'].unique())} valid trials (after filtering):")
    for tid in sorted(events_df["trial_id"].unique()):
        print(f"  → {tid}")
        plot_trial_with_episodes(SUBJECT_NAME, tid, events_df)
else:
    print("No episodes detected after filtering → check logs.")
    display(logs_df)

In [ ]:
# ─── Run cell ────────────────────────────────────────────────
SUBJECT_NAME = "ALS_Subject_10"   # <- change only this line

print(f"\n=== Starting Phase 4 processing for {SUBJECT_NAME} ===\n")
print(f"Current filter settings: min_duration={MIN_VALID_EPISODE_DURATION}s | min_peak_gate={MIN_VALID_PEAK_GATE}")

events_df, logs_df = run_phase4_for_subject(
    subject_name = SUBJECT_NAME,
    method       = "mad",
    k_on         = 5.0,
    k_off        = 3.50,
    pad_s        = 0.15,
    use_acc_gate = True,
    alpha_acc    = 1.0,
    compute_sanity_check = True,
)

# ─── Display summary results ────────────────────────────────
if not events_df.empty:
    print("\nEpisode duration distribution (seconds):")
    display(events_df["duration_s"].describe())
    
    print("\nNumber of episodes per trial and data source:")
    display(events_df.groupby(["trial_id", "data_source"])["episode_id"].count().unstack(fill_value=0))
    
    # new: summary of discarded episodes from logs_df (if the filter fired)
    if 'discarded_episodes' in logs_df.columns:
        total_discarded = logs_df["discarded_episodes"].sum()
        print(f"\nTotal discarded short/weak episodes across all trials: {total_discarded}")
        display(logs_df[logs_df["discarded_episodes"] > 0][["trial_id", "discarded_episodes", "task_type"]])
    
    print(f"\nPlotting {len(events_df['trial_id'].unique())} valid trials (after filtering):")
    for tid in sorted(events_df["trial_id"].unique()):
        print(f"  → {tid}")
        plot_trial_with_episodes(SUBJECT_NAME, tid, events_df)
else:
    print("No episodes detected after filtering → check logs.")
    display(logs_df)

In [ ]:
# ============================================================
# ALS_Subject_10 post-fix:
# 1) Drop ALL episodes of pick&place_high_exo (broken signal)
# 2) Drop LAST episode of lifting_exo (episode #4 by time order)
# 3) Save updated subject parquet
# 4) Plot both trials to verify
# ============================================================

SUBJECT_TO_FIX       = "ALS_Subject_10"
TRIAL_DROP_ALL       = "pick&place_high_exo"  # substring ok
TRIAL_DROP_LAST      = "lifting_exo"          # substring ok
SAVE_UPDATED         = True

# ---- basic checks ----
required = ["PHASE4_DIR", "plot_trial_with_episodes"]
missing = [x for x in required if x not in globals()]
if missing:
    raise NameError(f"Missing required: {missing}")

# Prefer using in-memory events_df if it matches subject; otherwise load from parquet
use_mem = ("events_df" in globals()) and (not events_df.empty) and (events_df["subject"].astype(str) == SUBJECT_TO_FIX).all()

ep_path = PHASE4_DIR / f"{SUBJECT_TO_FIX}__episodes.parquet"
if use_mem:
    df = events_df.copy()
else:
    if not ep_path.exists():
        raise FileNotFoundError(f"episodes parquet not found: {ep_path}")
    df = pd.read_parquet(ep_path)

df["trial_id"] = df["trial_id"].astype(str)

# ------------------------
# (1) Drop ALL episodes of pick&place_high_exo
# ------------------------
mask_all = df["trial_id"].str.contains(TRIAL_DROP_ALL, case=False, na=False)
matched_all = sorted(df.loc[mask_all, "trial_id"].unique().tolist())

removed_all = int(mask_all.sum())
df = df.loc[~mask_all].copy()

print(f"🗑️ Drop-all trial '{TRIAL_DROP_ALL}': removed {removed_all} rows. Matched trials: {matched_all if matched_all else 'None'}")

# ------------------------
# (2) Drop LAST episode of lifting_exo (by time order)
# ------------------------
mask_lift = df["trial_id"].str.contains(TRIAL_DROP_LAST, case=False, na=False)
matched_lift = sorted(df.loc[mask_lift, "trial_id"].unique().tolist())

if not matched_lift:
    print(f"⚠️ No trial matched '{TRIAL_DROP_LAST}' (nothing to drop last).")
else:
    if len(matched_lift) > 1:
        print("⚠️ Multiple lifting trials matched. Using first:", matched_lift)
    trial_lift_exact = matched_lift[0]

    df_lift = df[df["trial_id"] == trial_lift_exact].copy().sort_values(["t_on","t_off"]).reset_index()  # keep original idx
    if df_lift.empty:
        print(f"⚠️ lifting trial '{trial_lift_exact}' has no episodes (nothing to drop).")
    else:
        idx_drop = df_lift.iloc[-1]["index"]  # last by time order
        print("✅ Dropping LAST lifting episode row:")
        display(df.loc[idx_drop, ["subject","trial_id","episode_id","t_on","t_off","duration_s","peak_gate","data_source","task_type"]])

        df = df.drop(index=idx_drop).reset_index(drop=True)

        # Re-number episode_id within lifting trial for cleanliness
        df.loc[df["trial_id"] == trial_lift_exact, "episode_id"] = (
            df[df["trial_id"] == trial_lift_exact]
            .sort_values("t_on")
            .groupby("trial_id")
            .cumcount() + 1
        )

# ------------------------
# (3) Save updated parquet
# ------------------------
if SAVE_UPDATED:
    df.to_parquet(ep_path, index=False)
    print(f"\n💾 Updated parquet saved: {ep_path}")

# Sync back to in-memory events_df too (so downstream plots/cells use updated)
events_df = df

# ------------------------
# (4) Plot to verify
# ------------------------
# Plot lifting_exo (after dropping last)
if 'trial_lift_exact' in locals():
    print(f"\n📈 Plot after fixes → {SUBJECT_TO_FIX} / {trial_lift_exact}")
    plot_trial_with_episodes(SUBJECT_TO_FIX, trial_lift_exact, events_df)

# Plot pick&place_high_exo (should have 0 episodes now; likely no spans)
# We plot the matched trial_id(s) if any were found earlier; otherwise try the substring directly
if matched_all:
    for tid in matched_all:
        print(f"\n📈 Plot after fixes → {SUBJECT_TO_FIX} / {tid} (should have no episodes)")
        plot_trial_with_episodes(SUBJECT_TO_FIX, tid, events_df)
else:
    # attempt plot with the substring name (in case your loader uses exact string)
    print(f"\n📈 Plot after fixes → {SUBJECT_TO_FIX} / {TRIAL_DROP_ALL} (should have no episodes)")
    plot_trial_with_episodes(SUBJECT_TO_FIX, TRIAL_DROP_ALL, events_df)


In [ ]:
# =========================================
# Phase 4 - Activity Detection (Onset / Offset) - FINAL VERSION
# Tested and validated on ALS_Subject_13 - All 6 trials plotted correctly
# No over-merging, no missing bursts
# =========================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt

# ─── Paths ────────────────────────────────────────
NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed"
PHASE2B_DIR = PROCESSED_ROOT / "phase_02b_preprocess_imu"
PHASE3_DIR = PROCESSED_ROOT / "phase_03_normalization"
PHASE3_NORM_DIR = PHASE3_DIR / "normalized_env_per_subject"
PHASE4_DIR = PROCESSED_ROOT / "phase_04_events"
PHASE4_DIR.mkdir(parents=True, exist_ok=True)

# ─── Global thresholds (tuned for ALS) ─────────────────
MIN_VALID_EPISODE_DURATION = 3.5   # seconds - shorter episodes are discarded
MIN_VALID_PEAK_GATE = 28.0         # high sensitivity for weak bursts

# ─── Task-specific settings (tuned) ─────────────────
TASK_SETTINGS = {
    "drinking":   {"merge_gap_s": 0.35, "min_peak_gate": 28.0},
    "lifting":    {"merge_gap_s": 0.45, "min_peak_gate": 35.0},
    "pick&place": {"merge_gap_s": 0.50, "min_peak_gate": 32.0},
    "default":    {"merge_gap_s": 0.45, "min_peak_gate": 30.0}
}

# ─── Helper: low-pass filter ─────────────────────────────
def lowpass_filter(data, cutoff, fs, order=2):
    if len(data) < 10 or fs <= 0:
        return data
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    if normal_cutoff >= 1.0:
        return data
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    return filtfilt(b, a, data)

# ─── Episode detection with hysteresis (final stable version) ───────
def detect_episodes_hysteresis(
    t: np.ndarray,
    gate: np.ndarray,
    k_on: float = 4.8,
    k_off: float = 6.2,        # k_off > k_on -> much faster offset at pauses
    min_duration_s: float = 1.8,
    merge_gap_s: float = 0.4,
    pad_s: float = 0.15,
    task_key: str = "default"
) -> tuple[pd.DataFrame, dict]:
    t = np.asarray(t, dtype=float)
    gate = np.asarray(gate, dtype=float)
    mask = np.isfinite(t) & np.isfinite(gate)
    t, gate = t[mask], gate[mask]

    log = {
        "raw_samples": len(t),
        "gate_median": float(np.median(gate)),
        "gate_95p": float(np.percentile(gate, 95)),
    }

    if len(t) < 20:
        log["warning"] = "too_few_samples"
        return pd.DataFrame(), log

    # Baseline from the lowest 15% of values (robust to tremor and drift)
    baseline_perc = 15
    baseline_gate = gate[gate <= np.percentile(gate, baseline_perc)]
    if len(baseline_gate) < 10:
        baseline_gate = gate

    med = np.median(baseline_gate)
    mad = np.median(np.abs(baseline_gate - med)) + 1e-12

    thr_on = med + k_on * mad
    thr_off = med + k_off * mad   # thr_off above thr_on -> fast offset

    thr_off_multiplier = 1.1 if task_key in ["drinking", "lifting"] else 1.3
    thr_off *= thr_off_multiplier
    log["thr_off_multiplier"] = thr_off_multiplier
    log["thr_off_adjusted"] = float(thr_off)

    # cap to avoid an unreasonable threshold
    thr_on = min(thr_on, np.percentile(gate, 50))

    log.update({
        "thr_on": float(thr_on),
        "thr_off": float(thr_off),
        "baseline_perc_used": baseline_perc,
        "k_on": k_on,
        "k_off": k_off,
    })

    # Hysteresis detection
    episodes = []
    in_episode = False
    t_start = None
    for i in range(len(t)):
        if not in_episode:
            if gate[i] >= thr_on:
                in_episode = True
                t_start = t[i]
        else:
            if gate[i] <= thr_off:
                if t[i] - t_start >= 0.3:  # at least 300 ms to avoid noise
                    episodes.append([t_start, t[i]])
                in_episode = False
                t_start = None

    if in_episode and (t[-1] - t_start) >= 0.3:
        episodes.append([t_start, t[-1]])

    if not episodes:
        log["warning"] = "no_episodes_raw"
        return pd.DataFrame(), log

    # Padding and simple merging (no secondary / stronger merging)
    padded = [[max(t[0], a - pad_s), min(t[-1], b + pad_s)] for a, b in episodes]
    padded.sort(key=lambda x: x[0])

    merged = [padded[0]]
    for a, b in padded[1:]:
        if a - merged[-1][1] <= merge_gap_s:
            merged[-1][1] = max(merged[-1][1], b)
        else:
            merged.append([a, b])

    # build final_ep with the final filters
    final_ep = []
    for a, b in merged:
        dur = b - a
        if dur < min_duration_s:
            continue
        segment = gate[(t >= a) & (t <= b)]
        if len(segment) == 0:
            continue
        peak = np.max(segment)
        if peak >= MIN_VALID_PEAK_GATE:
            final_ep.append([a, b, dur, peak])

    df_ep = pd.DataFrame(final_ep, columns=["t_on", "t_off", "duration_s", "peak_gate"])
    log["n_episodes_final"] = len(df_ep)

    return df_ep, log

# ─── Main run function (final version) ───────────────────────────
def run_phase4_for_subject(
    subject_name: str,
    gate_lp_cutoff_imu: float = 7.0,
    gate_lp_cutoff_emg: float = 3.0,
):
    imu_path = PHASE2B_DIR / f"{subject_name}__imu_gate.parquet"
    emg_path = PHASE3_NORM_DIR / f"{subject_name}__emg_env_norm.parquet"

    df_imu = pd.read_parquet(imu_path) if imu_path.exists() else pd.DataFrame()
    df_emg = pd.read_parquet(emg_path) if emg_path.exists() else pd.DataFrame()

    if df_imu.empty and df_emg.empty:
        raise FileNotFoundError(f"No data for {subject_name}")

    all_trials = sorted(set(df_imu["trial_id"].unique()) | set(df_emg["trial_id"].unique()))
    print(f"Processing {subject_name} - {len(all_trials)} trials")

    events = []
    logs = []

    for trial_id in all_trials:
        use_imu = trial_id in df_imu["trial_id"].values
        df_trial = df_imu[df_imu["trial_id"] == trial_id] if use_imu else df_emg[df_emg["trial_id"] == trial_id]

        task_key = next((k for k in TASK_SETTINGS if k in trial_id.lower()), "default")
        settings = TASK_SETTINGS.get(task_key, TASK_SETTINGS["default"])

        print(f" → {trial_id} ({task_key}) - {'IMU' if use_imu else 'EMG fallback'}")

        t_col = "t_imu" if use_imu else "t_emg"
        t = df_trial[t_col].values

        if use_imu:
            gate_raw = np.maximum(df_trial["gyro_gate"].values, df_trial.get("acc_gate", 0).values)
            fs = 1 / np.mean(np.diff(t)) if len(t) > 1 else 148.0
            gate = lowpass_filter(gate_raw, gate_lp_cutoff_imu, fs)
        else:
            env_cols = [c for c in df_trial.columns if c.startswith("env_norm_")]
            gate_raw = df_trial[env_cols].max(axis=1).values
            fs = 1 / np.mean(np.diff(t)) if len(t) > 1 else 1259.0
            gate = lowpass_filter(gate_raw, gate_lp_cutoff_emg, fs)

        # special setting for lifting (fixes over-merging in lifting_exo)
        k_off_current = 6.2
        merge_gap_current = settings["merge_gap_s"]

        if task_key in ["drinking", "lifting"]:
            merge_gap_current = 1.2   # 1.2 s merging -> nearby bursts become one
            k_off_current = 5.0       # slightly lower to soften
            print(f" → {task_key} mode: merge_gap={merge_gap_current}s, k_off={k_off_current} (more merging, less segmentation)")

        ep_df, log = detect_episodes_hysteresis(
            t, gate,
            k_on=4.8,
            k_off=k_off_current,
            min_duration_s=settings.get("min_duration_s", 1.8),
            merge_gap_s=merge_gap_current,
            pad_s=0.15,
        )

        # global final filter
        if not ep_df.empty:
            ep_df = ep_df[ep_df["duration_s"] >= MIN_VALID_EPISODE_DURATION]
            ep_df = ep_df[ep_df["peak_gate"] >= MIN_VALID_PEAK_GATE]

        log.update({
            "subject": subject_name,
            "trial_id": trial_id,
            "task_type": task_key,
            "data_source": "imu" if use_imu else "emg",
        })
        logs.append(log)

        if not ep_df.empty:
            ep_df["trial_id"] = trial_id
            ep_df["subject"] = subject_name
            ep_df["task_type"] = task_key
            ep_df["episode_id"] = range(1, len(ep_df) + 1)
            events.append(ep_df)

    events_df = pd.concat(events, ignore_index=True) if events else pd.DataFrame()
    logs_df = pd.DataFrame(logs)

    events_df.to_parquet(PHASE4_DIR / f"{subject_name}__episodes.parquet", index=False)
    logs_df.to_csv(PHASE4_DIR / f"{subject_name}__episodes_log.csv", index=False)

    print(f"\n{subject_name} processed -> {len(events_df)} valid episodes from {len(all_trials)} trials")
    return events_df, logs_df

# ─── Plot function (episodes per trial) ────────────────
def plot_trial_with_episodes(subject_name, trial_id, episodes_df):
    imu_path = PHASE2B_DIR / f"{subject_name}__imu_gate.parquet"
    emg_path = PHASE3_NORM_DIR / f"{subject_name}__emg_env_norm.parquet"

    # filter valid episodes for this trial
    ep = episodes_df[
        (episodes_df["subject"] == subject_name) &
        (episodes_df["trial_id"] == trial_id) &
        (episodes_df["duration_s"] >= MIN_VALID_EPISODE_DURATION) &
        (episodes_df["peak_gate"] >= MIN_VALID_PEAK_GATE)
    ].copy().sort_values("t_on")

    if ep.empty:
        print(f" → No valid episodes for {trial_id} → skipping plot")
        return

    plt.figure(figsize=(14, 5))
    gate_source = "Unknown"

    # load the gate from IMU if available
    if imu_path.exists():
        df_imu = pd.read_parquet(imu_path)
        df_trial = df_imu[(df_imu["subject"] == subject_name) & (df_imu["trial_id"] == trial_id)]
        if not df_trial.empty:
            df_trial = df_trial.sort_values("t_imu")
            t = df_trial["t_imu"].values
            gyro = df_trial["gyro_gate"].values
            acc = df_trial.get("acc_gate", np.zeros_like(gyro)).values
            gate = np.maximum(gyro, 1.0 * acc)
            # smooth the gate for a cleaner plot
            fs = 1 / np.mean(np.diff(t)) if len(t) > 1 else 148.0
            gate = lowpass_filter(gate, 7.0, fs)
            plt.plot(t, gate, lw=1.3, label="gate: max(gyro, 1.0 × acc) (IMU)")
            gate_source = "IMU"

    # if no IMU, use the EMG fallback
    if gate_source == "Unknown" and emg_path.exists():
        df_emg = pd.read_parquet(emg_path)
        df_trial = df_emg[(df_emg["subject"] == subject_name) & (df_emg["trial_id"] == trial_id)]
        if not df_trial.empty:
            df_trial = df_trial.sort_values("t_emg")
            t = df_trial["t_emg"].values
            env_cols = [c for c in df_trial.columns if c.startswith("env_norm_")]
            if env_cols:
                gate = df_trial[env_cols].max(axis=1).values
                fs = 1 / np.mean(np.diff(t)) if len(t) > 1 else 1259.0
                gate = lowpass_filter(gate, 3.0, fs)
                plt.plot(t, gate, lw=1.3, label="gate: max(env_norm) (EMG fallback)")
                gate_source = "EMG fallback"

    if gate_source == "Unknown":
        print(f" → No gate data for {trial_id} → cannot plot")
        plt.close()
        return

    # plot the episodes
    for i, (_, r) in enumerate(ep.iterrows()):
        plt.axvspan(r["t_on"], r["t_off"], alpha=0.18, color='orange')
        plt.axvline(r["t_on"], ls="--", lw=1.1, color='darkgreen', label='on' if i == 0 else None)
        plt.axvline(r["t_off"], ls="--", lw=1.1, color='darkred', label='off' if i == 0 else None)

    plt.title(f"{subject_name} — {trial_id} — Episodes: {len(ep)}")
    plt.xlabel("Time (seconds)")
    plt.ylabel("Gate")
    plt.grid(True, alpha=0.35)
    plt.legend()
    plt.tight_layout()
    plt.show()
# ─── Run ───────────────────────────────────────
events_df, logs_df = run_phase4_for_subject(
    subject_name="ALS_Subject_13",
    gate_lp_cutoff_imu=7.0,
    gate_lp_cutoff_emg=3.0,
)

# show summary
if not events_df.empty:
    print("\nEpisode duration distribution (s):")
    display(events_df["duration_s"].describe())
    print(f"\nEpisodes per trial:")
    display(events_df["trial_id"].value_counts().sort_index())

# plot all trials
for tid in events_df["trial_id"].unique():
    plot_trial_with_episodes("ALS_Subject_13", tid, events_df)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt
# ─── Paths ────────────────────────────────────────
NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed"
PHASE2B_DIR = PROCESSED_ROOT / "phase_02b_preprocess_imu"
PHASE3_DIR = PROCESSED_ROOT / "phase_03_normalization"
PHASE3_NORM_DIR = PHASE3_DIR / "normalized_env_per_subject"
PHASE4_DIR = PROCESSED_ROOT / "phase_04_events"
PHASE4_DIR.mkdir(parents=True, exist_ok=True)
# ─── Global thresholds (tuned for ALS) ─────────────────
MIN_VALID_EPISODE_DURATION = 3.5 # seconds - shorter episodes are discarded
MIN_VALID_PEAK_GATE = 28.0 # high sensitivity for weak bursts
# ─── Task-specific settings (tuned) ─────────────────
TASK_SETTINGS = {
    "drinking": {"merge_gap_s": 0.35, "min_peak_gate": 28.0},
    "lifting": {"merge_gap_s": 0.45, "min_peak_gate": 35.0},
    "pick&place": {"merge_gap_s": 0.50, "min_peak_gate": 32.0},
    "default": {"merge_gap_s": 0.45, "min_peak_gate": 30.0}
}
# ─── Helper: low-pass filter ─────────────────────────────
def lowpass_filter(data, cutoff, fs, order=2):
    if len(data) < 10 or fs <= 0:
        return data
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    if normal_cutoff >= 1.0:
        return data
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    return filtfilt(b, a, data)
# ─── Episode detection with hysteresis (final stable version) ───────
def detect_episodes_hysteresis(
    t: np.ndarray,
    gate: np.ndarray,
    k_on: float = 4.8,
    k_off: float = 6.2, # k_off > k_on -> much faster offset at pauses
    min_duration_s: float = 1.8,
    merge_gap_s: float = 0.4,
    pad_s: float = 0.15,
    task_key: str = "default"
) -> tuple[pd.DataFrame, dict]:
    t = np.asarray(t, dtype=float)
    gate = np.asarray(gate, dtype=float)
    mask = np.isfinite(t) & np.isfinite(gate)
    t, gate = t[mask], gate[mask]
    log = {
        "raw_samples": len(t),
        "gate_median": float(np.median(gate)),
        "gate_95p": float(np.percentile(gate, 95)),
    }
    if len(t) < 20:
        log["warning"] = "too_few_samples"
        return pd.DataFrame(), log
    # Baseline from the lowest 15% of values (robust to tremor and drift)
    baseline_perc = 15
    baseline_gate = gate[gate <= np.percentile(gate, baseline_perc)]
    if len(baseline_gate) < 10:
        baseline_gate = gate
    med = np.median(baseline_gate)
    mad = np.median(np.abs(baseline_gate - med)) + 1e-12
    thr_on = med + k_on * mad
    thr_off = med + k_off * mad # thr_off above thr_on -> fast offset
    thr_off_multiplier = 1.1 if task_key in ["drinking", "lifting"] else 1.3
    thr_off *= thr_off_multiplier
    log["thr_off_multiplier"] = thr_off_multiplier
    log["thr_off_adjusted"] = float(thr_off)
    # cap to avoid an unreasonable threshold
    thr_on = min(thr_on, np.percentile(gate, 50))
    log.update({
        "thr_on": float(thr_on),
        "thr_off": float(thr_off),
        "baseline_perc_used": baseline_perc,
        "k_on": k_on,
        "k_off": k_off,
        "merge_gap_used": merge_gap_s,
    })
    # Hysteresis detection
    episodes = []
    in_episode = False
    t_start = None
    for i in range(len(t)):
        if not in_episode:
            if gate[i] >= thr_on:
                in_episode = True
                t_start = t[i]
        else:
            if gate[i] <= thr_off:
                if t[i] - t_start >= 0.3: # at least 300 ms to avoid noise
                    episodes.append([t_start, t[i]])
                in_episode = False
                t_start = None
    if in_episode and (t[-1] - t_start) >= 0.3:
        episodes.append([t_start, t[-1]])
    if not episodes:
        log["warning"] = "no_episodes_raw"
        return pd.DataFrame(), log
    # Padding and simple merging (no secondary / stronger merging)
    padded = [[max(t[0], a - pad_s), min(t[-1], b + pad_s)] for a, b in episodes]
    padded.sort(key=lambda x: x[0])
    merged = [padded[0]]
    for a, b in padded[1:]:
        if a - merged[-1][1] <= merge_gap_s:
            merged[-1][1] = max(merged[-1][1], b)
        else:
            merged.append([a, b])
    # build final_ep with the final filters
    final_ep = []
    for a, b in merged:
        dur = b - a
        if dur < min_duration_s:
            continue
        segment = gate[(t >= a) & (t <= b)]
        if len(segment) == 0:
            continue
        peak = np.max(segment)
        if peak >= MIN_VALID_PEAK_GATE:
            final_ep.append([a, b, dur, peak])
    df_ep = pd.DataFrame(final_ep, columns=["t_on", "t_off", "duration_s", "peak_gate"])
    log["n_episodes_final"] = len(df_ep)
    return df_ep, log
# ─── Main run function (final version) ───────────────────────────
def run_phase4_for_subject(
    subject_name: str,
    gate_lp_cutoff_imu: float = 7.0,
    gate_lp_cutoff_emg: float = 3.0,
):
    imu_path = PHASE2B_DIR / f"{subject_name}__imu_gate.parquet"
    emg_path = PHASE3_NORM_DIR / f"{subject_name}__emg_env_norm.parquet"
    df_imu = pd.read_parquet(imu_path) if imu_path.exists() else pd.DataFrame()
    df_emg = pd.read_parquet(emg_path) if emg_path.exists() else pd.DataFrame()
    if df_imu.empty and df_emg.empty:
        raise FileNotFoundError(f"No data for {subject_name}")
    all_trials = sorted(set(df_imu["trial_id"].unique()) | set(df_emg["trial_id"].unique()))
    print(f"Processing {subject_name} - {len(all_trials)} trials")
    events = []
    logs = []
    for trial_id in all_trials:
        use_imu = trial_id in df_imu["trial_id"].values
        df_trial = df_imu[df_imu["trial_id"] == trial_id] if use_imu else df_emg[df_emg["trial_id"] == trial_id]
        task_key = next((k for k in TASK_SETTINGS if k in trial_id.lower()), "default")
        settings = TASK_SETTINGS.get(task_key, TASK_SETTINGS["default"])
        print(f" → {trial_id} ({task_key}) - {'IMU' if use_imu else 'EMG fallback'}")
        t_col = "t_imu" if use_imu else "t_emg"
        t = df_trial[t_col].values
        if use_imu:
            gate_raw = np.maximum(df_trial["gyro_gate"].values, df_trial.get("acc_gate", 0).values)
            fs = 1 / np.mean(np.diff(t)) if len(t) > 1 else 148.0
            gate = lowpass_filter(gate_raw, gate_lp_cutoff_imu, fs)
        else:
            env_cols = [c for c in df_trial.columns if c.startswith("env_norm_")]
            gate_raw = df_trial[env_cols].max(axis=1).values
            fs = 1 / np.mean(np.diff(t)) if len(t) > 1 else 1259.0
            gate = lowpass_filter(gate_raw, gate_lp_cutoff_emg, fs)
        # special setting for lifting (fixes over-merging in lifting_exo)
        k_off_current = 6.2
        merge_gap_current = settings["merge_gap_s"]
        pad_s_current = 0.15
        curr_k_on = 4.8
        # special setting by exo/noexo
        if "noexo" in trial_id.lower():
            k_off_current = 7.0 # higher for more segmentation (easier offset)
            merge_gap_current = 0.3 # smaller for less merging
            print(f" → noexo mode: merge_gap={merge_gap_current}s, k_off={k_off_current} (more segmentation, less merging)")
        elif "exo" in trial_id.lower():
            k_off_current = 5.0 # lower for less segmentation (harder offset)
            merge_gap_current = 0.8 # higher for more merging (merge small dips)
            print(f" → exo mode: merge_gap={merge_gap_current}s, k_off={k_off_current} (less segmentation, more merging)")
        # ─── new: targeted setting against over-merging in pick&place_high_exo ───
        if task_key == "pick&place" and "high_exo" in trial_id.lower():
            k_off_current = 9.5          # higher -> faster offset at rest dips
            merge_gap_current = 0.15     # very small -> even medium gaps split episodes
            pad_s_current = 0.08         # small margin -> less stretching over the dip
            curr_k_on = 5.2              # slightly above baseline for a firmer onset in the second part
            print(f" → special high_exo pick&place mode: merge_gap={merge_gap_current}s, k_off={k_off_current} (targeted more segmentation, less merging to fix over-merging)")
             # replaces the previous condition
        if task_key == "pick&place" and "high" in trial_id.lower() and "noexo" in trial_id.lower():
            k_off_current = 8.5
            merge_gap_current = 0.25
            pad_s_current = 0.08
            curr_k_on = 5.2
            print(f" → ACTIVATED special pick&place_high_noexo mode for {trial_id}")
            print(f"    k_off={k_off_current}, merge_gap={merge_gap_current}s, pad_s={pad_s_current}, k_on={curr_k_on}")
        if task_key == "lifting" and "noexo" in trial_id.lower():
            merge_gap_current = 1.0 # above 0.3 -> merge the dip between episodes 1 and 2
            k_off_current = 9.5 # slightly higher -> harder offset, longer episodes (esp. the last)
            pad_s_current = 0.5 # larger margin to cover onset/offset fluctuations
            print(f" → special lifting_noexo mode: merge_gap={merge_gap_current}s, "
                f"k_off={k_off_current}, pad_s={pad_s_current} (longer episodes + better merging)")
        # for plain lifting (no special exo/noexo rule) enforce a minimum merge_gap
        if task_key == "lifting":
            merge_gap_current = max(merge_gap_current, 0.55) # minimum for lifting
        # ─── special settings for observed problems ───
        if task_key == "drinking" and "noexo" in trial_id.lower():
            k_off_current = 8.5  # higher for an easier offset (earlier end in the rest phase)
            print(f" → special drinking_noexo: k_off={k_off_current} (easier offset for earlier end in middle episode)")
        if task_key == "lifting" and "exo" in trial_id.lower():
            curr_k_on = 6.2          # was 5.5 -> now harder (later onset)
            k_off_current = 5.8      # slightly higher than before for a faster offset at middle dips
            merge_gap_current = 0.6  # medium -> neither over-merge nor fragment
            pad_s_current = 0.10     # smaller pad at onset to avoid stretching backwards
            print(f" → special lifting_exo: k_on={curr_k_on} (harder onset for later start in middle episode)")
        if task_key == "lifting" and "noexo" in trial_id.lower():
            k_off_current = 4.5  # lower for harder offset (longer episode, covers the preceding part in the third)
            merge_gap_current = 1.5  # higher for more merging (capture the preceding movement part)
            print(f" → adjusted lifting_noexo: merge_gap={merge_gap_current}s, k_off={k_off_current} (harder offset + more merging for complete third episode)")
        ep_df, log = detect_episodes_hysteresis(
            t, gate,
            k_on=curr_k_on,
            k_off=k_off_current,
            min_duration_s=settings.get("min_duration_s", 1.8),
            merge_gap_s=merge_gap_current,
            pad_s=0.15,
        )
        # global final filter
        if not ep_df.empty:
            ep_df = ep_df[ep_df["duration_s"] >= MIN_VALID_EPISODE_DURATION]
            ep_df = ep_df[ep_df["peak_gate"] >= MIN_VALID_PEAK_GATE]
            # ─── Special post-processing to remove small/weak unwanted episodes in lifting_exo ───
        if not ep_df.empty and task_key == "lifting" and "exo" in trial_id.lower():
            before_count = len(ep_df)
            
            # rule 1: drop very short episodes (< 2 s - usually noise or weak pre-movement)
            ep_df = ep_df[ep_df["duration_s"] >= 2.0].copy()
            
            # rule 2: if more than 3 episodes remain, drop the weakest (lowest peak)
            if len(ep_df) > 3:
                # sort by peak_gate ascending -> weakest first
                ep_df = ep_df.sort_values("peak_gate").iloc[1:].reset_index(drop=True)
                print(f"  → Removed weakest/short episode in {trial_id} (now {len(ep_df)} episodes)")
            
            # rule 3: drop the 2nd episode if it is very short/weak (position-based)
            if len(ep_df) >= 3:
                second_ep = ep_df.iloc[1]  # second episode (index 1)
                if second_ep["duration_s"] < 2.5 or second_ep["peak_gate"] < 120:  # empirical threshold
                    ep_df = ep_df.drop(ep_df.index[1]).reset_index(drop=True)
                    print(f"  → Removed suspicious second episode in {trial_id} (short/weak middle rest)")
            
            after_count = len(ep_df)
            if after_count < before_count:
                print(f"  → Post-processing removed {before_count - after_count} episode(s) in lifting_exo")
        log.update({
            "subject": subject_name,
            "trial_id": trial_id,
            "task_type": task_key,
            "data_source": "imu" if use_imu else "emg",
        })
        logs.append(log)
        if not ep_df.empty:
            ep_df["trial_id"] = trial_id
            ep_df["subject"] = subject_name
            ep_df["task_type"] = task_key
            ep_df["episode_id"] = range(1, len(ep_df) + 1)
            events.append(ep_df)
    events_df = pd.concat(events, ignore_index=True) if events else pd.DataFrame()
    logs_df = pd.DataFrame(logs)
    events_df.to_parquet(PHASE4_DIR / f"{subject_name}__episodes.parquet", index=False)
    logs_df.to_csv(PHASE4_DIR / f"{subject_name}__episodes_log.csv", index=False)
    print(f"\n{subject_name} processed -> {len(events_df)} valid episodes from {len(all_trials)} trials")
    return events_df, logs_df
# ─── Plot function (episodes per trial) ────────────────
def plot_trial_with_episodes(subject_name, trial_id, episodes_df):
    imu_path = PHASE2B_DIR / f"{subject_name}__imu_gate.parquet"
    emg_path = PHASE3_NORM_DIR / f"{subject_name}__emg_env_norm.parquet"
    # filter valid episodes for this trial
    ep = episodes_df[
        (episodes_df["subject"] == subject_name) &
        (episodes_df["trial_id"] == trial_id) &
        (episodes_df["duration_s"] >= MIN_VALID_EPISODE_DURATION) &
        (episodes_df["peak_gate"] >= MIN_VALID_PEAK_GATE)
    ].copy().sort_values("t_on")
    if ep.empty:
        print(f" → No valid episodes for {trial_id} → skipping plot")
        return
    plt.figure(figsize=(14, 5))
    gate_source = "Unknown"
    # load the gate from IMU if available
    if imu_path.exists():
        df_imu = pd.read_parquet(imu_path)
        df_trial = df_imu[(df_imu["subject"] == subject_name) & (df_imu["trial_id"] == trial_id)]
        if not df_trial.empty:
            df_trial = df_trial.sort_values("t_imu")
            t = df_trial["t_imu"].values
            gyro = df_trial["gyro_gate"].values
            acc = df_trial.get("acc_gate", np.zeros_like(gyro)).values
            gate = np.maximum(gyro, 1.0 * acc)
            # smooth the gate for a cleaner plot
            fs = 1 / np.mean(np.diff(t)) if len(t) > 1 else 148.0
            gate = lowpass_filter(gate, 7.0, fs)
            plt.plot(t, gate, lw=1.3, label="gate: max(gyro, 1.0 × acc) (IMU)")
            gate_source = "IMU"
    # if no IMU, use the EMG fallback
    if gate_source == "Unknown" and emg_path.exists():
        df_emg = pd.read_parquet(emg_path)
        df_trial = df_emg[(df_emg["subject"] == subject_name) & (df_emg["trial_id"] == trial_id)]
        if not df_trial.empty:
            df_trial = df_trial.sort_values("t_emg")
            t = df_trial["t_emg"].values
            env_cols = [c for c in df_trial.columns if c.startswith("env_norm_")]
            if env_cols:
                gate = df_trial[env_cols].max(axis=1).values
                fs = 1 / np.mean(np.diff(t)) if len(t) > 1 else 1259.0
                gate = lowpass_filter(gate, 3.0, fs)
                plt.plot(t, gate, lw=1.3, label="gate: max(env_norm) (EMG fallback)")
                gate_source = "EMG fallback"
    if gate_source == "Unknown":
        print(f" → No gate data for {trial_id} → cannot plot")
        plt.close()
        return
    # plot the episodes
    for i, (_, r) in enumerate(ep.iterrows()):
        plt.axvspan(r["t_on"], r["t_off"], alpha=0.18, color='orange')
        plt.axvline(r["t_on"], ls="--", lw=1.1, color='darkgreen', label='on' if i == 0 else None)
        plt.axvline(r["t_off"], ls="--", lw=1.1, color='darkred', label='off' if i == 0 else None)
    plt.title(f"{subject_name} — {trial_id} — Episodes: {len(ep)}")
    plt.xlabel("Time (seconds)")
    plt.ylabel("Gate")
    plt.grid(True, alpha=0.35)
    plt.legend()
    plt.tight_layout()
    plt.show()
# ─── Run ───────────────────────────────────────
events_df, logs_df = run_phase4_for_subject(
    subject_name="ALS_Subject_12",
    gate_lp_cutoff_imu=7.0,
    gate_lp_cutoff_emg=3.0,
)
# show summary
if not events_df.empty:
    print("\nEpisode duration distribution (s):")
    display(events_df["duration_s"].describe())
    print(f"\nEpisodes per trial:")
    display(events_df["trial_id"].value_counts().sort_index())
# plot all trials
for tid in events_df["trial_id"].unique():
    plot_trial_with_episodes("ALS_Subject_12", tid, events_df)

In [ ]:
# ─── Global thresholds (tuned for ALS) ─────────────────
MIN_VALID_EPISODE_DURATION = 3.5   # seconds - shorter episodes are discarded
MIN_VALID_PEAK_GATE = 28.0         # high sensitivity for weak bursts

# ─── Task-specific settings (tuned) ─────────────────
TASK_SETTINGS = {
    "drinking": {"merge_gap_s": 0.35, "min_peak_gate": 28.0},
    "lifting": {"merge_gap_s": 0.45, "min_peak_gate": 35.0},
    "pick&place": {"merge_gap_s": 0.50, "min_peak_gate": 32.0},
    "default": {"merge_gap_s": 0.45, "min_peak_gate": 30.0}
}

# ─── Helper: low-pass filter ─────────────────────────────
def lowpass_filter(data, cutoff, fs, order=2):
    if len(data) < 10 or fs <= 0:
        return data
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    if normal_cutoff >= 1.0:
        return data
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    return filtfilt(b, a, data)

# ─── Episode detection with hysteresis (final stable version) ───────
def detect_episodes_hysteresis(
    t: np.ndarray,
    gate: np.ndarray,
    k_on: float = 4.8,
    k_off: float = 6.2,        # k_off > k_on -> much faster offset at pauses
    min_duration_s: float = 1.8,
    merge_gap_s: float = 0.4,
    pad_s: float = 0.15,
    task_key: str = "default"
) -> tuple[pd.DataFrame, dict]:
    t = np.asarray(t, dtype=float)
    gate = np.asarray(gate, dtype=float)
    mask = np.isfinite(t) & np.isfinite(gate)
    t, gate = t[mask], gate[mask]

    log = {
        "raw_samples": len(t),
        "gate_median": float(np.median(gate)),
        "gate_95p": float(np.percentile(gate, 95)),
    }

    if len(t) < 20:
        log["warning"] = "too_few_samples"
        return pd.DataFrame(), log

    # Baseline from the lowest 15% of values (robust to tremor and drift)
    baseline_perc = 15
    baseline_gate = gate[gate <= np.percentile(gate, baseline_perc)]
    if len(baseline_gate) < 10:
        baseline_gate = gate

    med = np.median(baseline_gate)
    mad = np.median(np.abs(baseline_gate - med)) + 1e-12

    thr_on = med + k_on * mad
    thr_off = med + k_off * mad   # thr_off above thr_on -> fast offset

    thr_off_multiplier = 1.1 if task_key in ["drinking", "lifting"] else 1.3
    thr_off *= thr_off_multiplier
    log["thr_off_multiplier"] = thr_off_multiplier
    log["thr_off_adjusted"] = float(thr_off)

    # cap to avoid an unreasonable threshold
    thr_on = min(thr_on, np.percentile(gate, 50))

    log.update({
        "thr_on": float(thr_on),
        "thr_off": float(thr_off),
        "baseline_perc_used": baseline_perc,
        "k_on": k_on,
        "k_off": k_off,
        "merge_gap_used": merge_gap_s,
    })

    # Hysteresis detection
    episodes = []
    in_episode = False
    t_start = None
    for i in range(len(t)):
        if not in_episode:
            if gate[i] >= thr_on:
                in_episode = True
                t_start = t[i]
        else:
            if gate[i] <= thr_off:
                if t[i] - t_start >= 0.3:  # at least 300 ms to avoid noise
                    episodes.append([t_start, t[i]])
                in_episode = False
                t_start = None

    if in_episode and (t[-1] - t_start) >= 0.3:
        episodes.append([t_start, t[-1]])

    if not episodes:
        log["warning"] = "no_episodes_raw"
        return pd.DataFrame(), log

    # Padding and simple merging (no secondary / stronger merging)
    padded = [[max(t[0], a - pad_s), min(t[-1], b + pad_s)] for a, b in episodes]
    padded.sort(key=lambda x: x[0])

    merged = [padded[0]]
    for a, b in padded[1:]:
        if a - merged[-1][1] <= merge_gap_s:
            merged[-1][1] = max(merged[-1][1], b)
        else:
            merged.append([a, b])

    # build final_ep with the final filters
    final_ep = []
    for a, b in merged:
        dur = b - a
        if dur < min_duration_s:
            continue
        segment = gate[(t >= a) & (t <= b)]
        if len(segment) == 0:
            continue
        peak = np.max(segment)
        if peak >= MIN_VALID_PEAK_GATE:
            final_ep.append([a, b, dur, peak])

    df_ep = pd.DataFrame(final_ep, columns=["t_on", "t_off", "duration_s", "peak_gate"])
    log["n_episodes_final"] = len(df_ep)

    return df_ep, log

# ─── Main run function (final version) ───────────────────────────
def run_phase4_for_subject(
    subject_name: str,
    gate_lp_cutoff_imu: float = 7.0,
    gate_lp_cutoff_emg: float = 3.0,
):
    imu_path = PHASE2B_DIR / f"{subject_name}__imu_gate.parquet"
    emg_path = PHASE3_NORM_DIR / f"{subject_name}__emg_env_norm.parquet"

    df_imu = pd.read_parquet(imu_path) if imu_path.exists() else pd.DataFrame()
    df_emg = pd.read_parquet(emg_path) if emg_path.exists() else pd.DataFrame()

    if df_imu.empty and df_emg.empty:
        raise FileNotFoundError(f"No data for {subject_name}")

    all_trials = sorted(set(df_imu["trial_id"].unique()) | set(df_emg["trial_id"].unique()))
    print(f"Processing {subject_name} - {len(all_trials)} trials")

    events = []
    logs = []

    for trial_id in all_trials:
        use_imu = trial_id in df_imu["trial_id"].values
        df_trial = df_imu[df_imu["trial_id"] == trial_id] if use_imu else df_emg[df_emg["trial_id"] == trial_id]

        task_key = next((k for k in TASK_SETTINGS if k in trial_id.lower()), "default")
        settings = TASK_SETTINGS.get(task_key, TASK_SETTINGS["default"])

        print(f" → {trial_id} ({task_key}) - {'IMU' if use_imu else 'EMG fallback'}")

        t_col = "t_imu" if use_imu else "t_emg"
        t = df_trial[t_col].values

        if use_imu:
            gate_raw = np.maximum(df_trial["gyro_gate"].values, df_trial.get("acc_gate", 0).values)
            fs = 1 / np.mean(np.diff(t)) if len(t) > 1 else 148.0
            gate = lowpass_filter(gate_raw, gate_lp_cutoff_imu, fs)
        else:
            env_cols = [c for c in df_trial.columns if c.startswith("env_norm_")]
            gate_raw = df_trial[env_cols].max(axis=1).values
            fs = 1 / np.mean(np.diff(t)) if len(t) > 1 else 1259.0
            gate = lowpass_filter(gate_raw, gate_lp_cutoff_emg, fs)

        # special setting for lifting (fixes over-merging in lifting_exo)
        k_off_current = 6.2
        merge_gap_current = settings["merge_gap_s"]
        pad_s_current = 0.15

        # special setting by exo/noexo
        if "noexo" in trial_id.lower():
            k_off_current = 7.0  # higher for more segmentation (easier offset)
            merge_gap_current = 0.3  # smaller for less merging
            print(f" → noexo mode: merge_gap={merge_gap_current}s, k_off={k_off_current} (more segmentation, less merging)")
        elif "exo" in trial_id.lower():
            k_off_current = 5.0  # lower for less segmentation (harder offset)
            merge_gap_current = 0.8  # higher for more merging (merge small dips)
            print(f" → exo mode: merge_gap={merge_gap_current}s, k_off={k_off_current} (less segmentation, more merging)")

        # ─── new: targeted setting against over-merging in pick&place_high_exo ───
        if task_key == "pick&place" and "high_exo" in trial_id.lower():
            k_off_current = 6.5  # slightly above the exo default (5.0) for more segmentation at dips (easier offset)
            merge_gap_current = 0.5  # below the exo default (0.8) for less merging (only tiny gaps merged)
            print(f" → special high_exo pick&place mode: merge_gap={merge_gap_current}s, k_off={k_off_current} (targeted more segmentation, less merging to fix over-merging)")

        if task_key == "lifting" and "noexo" in trial_id.lower():
            merge_gap_current = 1.0          # above 0.3 -> merge the dip between episodes 1 and 2
            k_off_current = 9.5              # slightly higher -> harder offset, longer episodes (esp. the last)
            pad_s_current = 0.5             # larger margin to cover onset/offset fluctuations
            print(f" → special lifting_noexo mode: merge_gap={merge_gap_current}s, "
                f"k_off={k_off_current}, pad_s={pad_s_current} (longer episodes + better merging)")
        # ─── special setting to shorten the first episode of drinking_exo (subject 11) ───
        if task_key == "drinking" and "exo" in trial_id.lower() and subject_name == "ALS_Subject_11":
            k_off_current = 9.5          # much higher -> faster offset at the rest dip
            merge_gap_current = 0.20     # small -> less merging
            pad_s_current = 0.08         # small margin
            print(f" → special drinking_exo for subj11: k_off={k_off_current}, merge_gap={merge_gap_current}s, pad_s={pad_s_current} (faster offset to shorten first episode)")
        # ─── new: targeted setting for over-merging in drinking_noexo ───
        if task_key == "drinking" and "noexo" in trial_id.lower():
            k_off_current = 10  # higher for more segmentation (easier offset at deep dips)
            merge_gap_current = 0.15  # smaller for less merging (only tiny gaps are merged)
            pad_s_current = 0.08  # smaller margin
            print(f" → special drinking_noexo for subj11: merge_gap={merge_gap_current}s, k_off={k_off_current}, pad_s={pad_s_current} (targeted more segmentation to fix over-merging)")

        # ─── new: targeted setting for over-merging in lifting_exo ───
        if task_key == "lifting" and "exo" in trial_id.lower():
            k_off_current = 6.5  # slightly above the exo default (5.0) for more segmentation
            merge_gap_current = 0.5  # below the default (0.8) for less merging
            pad_s_current = 0.12  # small margin
            print(f" → special lifting_exo for subj11: merge_gap={merge_gap_current}s, k_off={k_off_current}, pad_s={pad_s_current} (more segmentation, less merging to fix over-merging)")
        

        ep_df, log = detect_episodes_hysteresis(
            t, gate,
            k_on=4.8,
            k_off=k_off_current,
            min_duration_s=settings.get("min_duration_s", 1.8),
            merge_gap_s=merge_gap_current,
            pad_s=0.15,
        )

        # global final filter
        if not ep_df.empty:
            ep_df = ep_df[ep_df["duration_s"] >= MIN_VALID_EPISODE_DURATION]
            ep_df = ep_df[ep_df["peak_gate"] >= MIN_VALID_PEAK_GATE]
            # ─── new: do not save episodes of the corrupted lifting_exo trial ───
            if "lifting_exo" in trial_id.lower():
                ep_df = pd.DataFrame()

        log.update({
            "subject": subject_name,
            "trial_id": trial_id,
            "task_type": task_key,
            "data_source": "imu" if use_imu else "emg",
        })
        logs.append(log)

        if not ep_df.empty:
            ep_df["trial_id"] = trial_id
            ep_df["subject"] = subject_name
            ep_df["task_type"] = task_key
            ep_df["episode_id"] = range(1, len(ep_df) + 1)
            events.append(ep_df)

    events_df = pd.concat(events, ignore_index=True) if events else pd.DataFrame()
    logs_df = pd.DataFrame(logs)

    events_df.to_parquet(PHASE4_DIR / f"{subject_name}__episodes.parquet", index=False)
    logs_df.to_csv(PHASE4_DIR / f"{subject_name}__episodes_log.csv", index=False)

    print(f"\n{subject_name} processed -> {len(events_df)} valid episodes from {len(all_trials)} trials")
    return events_df, logs_df

# ─── Plot function (episodes per trial) ────────────────
def plot_trial_with_episodes(subject_name, trial_id, episodes_df):
    imu_path = PHASE2B_DIR / f"{subject_name}__imu_gate.parquet"
    emg_path = PHASE3_NORM_DIR / f"{subject_name}__emg_env_norm.parquet"

    # filter valid episodes for this trial
    ep = episodes_df[
        (episodes_df["subject"] == subject_name) &
        (episodes_df["trial_id"] == trial_id) &
        (episodes_df["duration_s"] >= MIN_VALID_EPISODE_DURATION) &
        (episodes_df["peak_gate"] >= MIN_VALID_PEAK_GATE)
    ].copy().sort_values("t_on")

    if ep.empty:
        print(f" → No valid episodes for {trial_id} → skipping plot")
        return

    plt.figure(figsize=(14, 5))
    gate_source = "Unknown"

    # load the gate from IMU if available
    if imu_path.exists():
        df_imu = pd.read_parquet(imu_path)
        df_trial = df_imu[(df_imu["subject"] == subject_name) & (df_imu["trial_id"] == trial_id)]
        if not df_trial.empty:
            df_trial = df_trial.sort_values("t_imu")
            t = df_trial["t_imu"].values
            gyro = df_trial["gyro_gate"].values
            acc = df_trial.get("acc_gate", np.zeros_like(gyro)).values
            gate = np.maximum(gyro, 1.0 * acc)
            # smooth the gate for a cleaner plot
            fs = 1 / np.mean(np.diff(t)) if len(t) > 1 else 148.0
            gate = lowpass_filter(gate, 7.0, fs)
            plt.plot(t, gate, lw=1.3, label="gate: max(gyro, 1.0 × acc) (IMU)")
            gate_source = "IMU"

    # if no IMU, use the EMG fallback
    if gate_source == "Unknown" and emg_path.exists():
        df_emg = pd.read_parquet(emg_path)
        df_trial = df_emg[(df_emg["subject"] == subject_name) & (df_emg["trial_id"] == trial_id)]
        if not df_trial.empty:
            df_trial = df_trial.sort_values("t_emg")
            t = df_trial["t_emg"].values
            env_cols = [c for c in df_trial.columns if c.startswith("env_norm_")]
            if env_cols:
                gate = df_trial[env_cols].max(axis=1).values
                fs = 1 / np.mean(np.diff(t)) if len(t) > 1 else 1259.0
                gate = lowpass_filter(gate, 3.0, fs)
                plt.plot(t, gate, lw=1.3, label="gate: max(env_norm) (EMG fallback)")
                gate_source = "EMG fallback"

    if gate_source == "Unknown":
        print(f" → No gate data for {trial_id} → cannot plot")
        plt.close()
        return

    # plot the episodes
    for i, (_, r) in enumerate(ep.iterrows()):
        plt.axvspan(r["t_on"], r["t_off"], alpha=0.18, color='orange')
        plt.axvline(r["t_on"], ls="--", lw=1.1, color='darkgreen', label='on' if i == 0 else None)
        plt.axvline(r["t_off"], ls="--", lw=1.1, color='darkred', label='off' if i == 0 else None)

    plt.title(f"{subject_name} — {trial_id} — Episodes: {len(ep)}")
    plt.xlabel("Time (seconds)")
    plt.ylabel("Gate")
    plt.grid(True, alpha=0.35)
    plt.legend()
    plt.tight_layout()
    plt.show()
# ─── Run ───────────────────────────────────────
events_df, logs_df = run_phase4_for_subject(
    subject_name="ALS_Subject_11",
    gate_lp_cutoff_imu=7.0,
    gate_lp_cutoff_emg=3.0,
)

# show summary
if not events_df.empty:
    print("\nEpisode duration distribution (s):")
    display(events_df["duration_s"].describe())
    print(f"\nEpisodes per trial:")
    display(events_df["trial_id"].value_counts().sort_index())

# plot all trials
for tid in events_df["trial_id"].unique():
    plot_trial_with_episodes("ALS_Subject_11", tid, events_df)

In [ ]:
# =========================================
# Phase 4 - Activity Detection (Onset / Offset) - Full version with EMG fallback
# Last update: 2026-02-08 (improved fallback for ALS cases)
# =========================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ─── Paths ────────────────────────────────────────
NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed"

PHASE2_DIR   = PROCESSED_ROOT / "phase_02_preprocess_emg"
PHASE2B_DIR  = PROCESSED_ROOT / "phase_02b_preprocess_imu"
PHASE3_DIR   = PROCESSED_ROOT / "phase_03_normalization"
PHASE3_NORM_DIR = PHASE3_DIR / "normalized_env_per_subject"  # fixed: exact path for env_norm
PHASE4_DIR   = PROCESSED_ROOT / "phase_04_events"

PHASE4_DIR.mkdir(parents=True, exist_ok=True)

print("Paths and initial settings loaded.")

# ─── Task-specific settings ─────────────────────────
TASK_SETTINGS = {
    "lifting":    {"min_duration_s": 1.8, "merge_gap_s": 0.35, "min_peak_gate": 100.0},
    "drinking":   {"min_duration_s": 1.2, "merge_gap_s": 0.30, "min_peak_gate": 60.0},
    "pick&place": {"min_duration_s": 2.2, "merge_gap_s": 1.20, "min_peak_gate": 75.0},
    "default":    {"min_duration_s": 1.5, "merge_gap_s": 0.40, "min_peak_gate": 70.0}
}

# Settings for EMG fallback mode (normalized envelope)
EMG_FALLBACK_SETTINGS = {
    "k_on": 2.0,                    # lower for better detection of weak ALS bursts
    "k_off": 1.0,                   # much lower → longer episode retention
    "peak_multiplier": 1.2,         # accept smaller peaks
    "min_duration_factor": 0.6,     # reduces min_duration_s by 40%
    "merge_gap_factor": 2.0,        # doubles merge_gap_s
    "secondary_merge_gap": 3.5      # much stronger merging to avoid fragmentation
}
# Minimum episode duration to keep (in seconds) - shorter episodes will be discarded as non-realistic
MIN_VALID_EPISODE_DURATION = 3.50  # Adjust based on visual inspection (e.g. 0.5 to 1.0 s)
MIN_VALID_PEAK_GATE = 50.0
# ─── Episode detection with hysteresis and post-processing ────────
def detect_episodes_hysteresis(
    t: np.ndarray,
    gate: np.ndarray,
    method: str = "mad",
    k_on: float = 5.0,
    k_off: float = 3.5,
    min_duration_s: float = 1.8,
    merge_gap_s: float = 0.45,
    pad_s: float = 0.15,
    peak_multiplier: float = 2.5,
    secondary_merge_gap: float = 0.8,
) -> tuple[pd.DataFrame, dict]:
    
    t   = np.asarray(t, dtype=float)
    gate = np.asarray(gate, dtype=float)

    mask_valid = np.isfinite(t) & np.isfinite(gate)
    t   = t[mask_valid]
    gate = gate[mask_valid]

    log = {
        "raw_samples": len(t),
        "gate_median": float(np.median(gate)) if len(gate) > 0 else np.nan,
        "gate_95p": float(np.percentile(gate, 95)) if len(gate) > 0 else np.nan,
    }

    if len(t) < 20:
        log["warning"] = "too_few_samples"
        return pd.DataFrame(columns=["t_on","t_off","duration_s","thr_on","thr_off","peak_gate"]), log

    med = float(np.median(gate))
    mad = float(np.median(np.abs(gate - med))) + 1e-12
    thr_on  = med + k_on  * mad
    thr_off = med + k_off * mad

    log["thr_on"]  = float(thr_on)
    log["thr_off"] = float(thr_off)

    # Fix: if thr_on too high (above 95p), automatically reduce k
    if thr_on > log["gate_95p"]:
        k_on = max(1.0, k_on * 0.7)  # reduce by 30%
        thr_on = med + k_on * mad
        log["thr_on_adjusted"] = thr_on
        log["warning"] = "thr_adjusted_down"

    episodes = []
    in_episode = False
    t_start = None

    for i in range(len(t)):
        if not in_episode:
            if gate[i] >= thr_on:
                in_episode = True
                t_start = float(t[i])
        else:
            if gate[i] <= thr_off:
                episodes.append([t_start, float(t[i])])
                in_episode = False
                t_start = None

    if in_episode:
        episodes.append([t_start, float(t[-1])])
        log["last_episode_open"] = True

    if not episodes:
        log["warning"] = "no_episodes_detected"
        return pd.DataFrame(columns=["t_on","t_off","duration_s","thr_on","thr_off","peak_gate"]), log

    t_min, t_max = float(t[0]), float(t[-1])
    padded = [[max(t_min, a - pad_s), min(t_max, b + pad_s)] for a, b in episodes]

    padded.sort(key=lambda x: x[0])
    merged = [padded[0]]
    for curr in padded[1:]:
        if curr[0] - merged[-1][1] <= merge_gap_s:
            merged[-1][1] = max(merged[-1][1], curr[1])
        else:
            merged.append(curr)

    final_merged = []
    current = merged[0]
    for nxt in merged[1:]:
        if nxt[0] - current[1] <= secondary_merge_gap:
            current[1] = max(current[1], nxt[1])
        else:
            final_merged.append(current)
            current = nxt
    final_merged.append(current)

    # Stronger merging for close intervals (< 1.2 s gap)
    secondary_merged = []
    current = final_merged[0] if final_merged else None
    for nxt in final_merged[1:]:
        if nxt[0] - current[1] <= 1.2:
            current[1] = max(current[1], nxt[1])
        else:
            secondary_merged.append(current)
            current = nxt
    if current:
        secondary_merged.append(current)

    final_merged = secondary_merged

    adaptive_peak_thr = med + peak_multiplier * mad
    log["adaptive_peak_thr"] = adaptive_peak_thr

    final_ep = []
    for a, b in final_merged:
        dur = b - a
        if dur < min_duration_s:
            continue

        segment_gate = gate[(t >= a) & (t <= b)]
        if len(segment_gate) == 0:
            continue
        peak = float(np.max(segment_gate))

        if peak >= adaptive_peak_thr:
            final_ep.append([a, b, dur, thr_on, thr_off, peak])

    df_ep = pd.DataFrame(final_ep, columns=["t_on", "t_off", "duration_s", "thr_on", "thr_off", "peak_gate"])
    log["n_episodes_after_postproc"] = len(df_ep)

    return df_ep, log


# ─── Build gate from IMU ────────────────────────────────────────
def build_gate(
    df_imu_trial: pd.DataFrame,
    use_acc_gate: bool = True,
    alpha: float = 1.0,
    time_bin_res: float = 0.001,
) -> tuple:
    df = df_imu_trial.copy()

    df["t_bin"] = (df["t_imu"] / time_bin_res).round().astype(int)

    agg_dict = {"t_imu": "mean", "gyro_gate": "median"}
    if use_acc_gate and "acc_gate" in df.columns:
        agg_dict["acc_gate"] = "median"

    df_agg = df.groupby("t_bin", as_index=False).agg(agg_dict).sort_values("t_imu")

    t    = df_agg["t_imu"].to_numpy(dtype=float)
    gyro = df_agg["gyro_gate"].to_numpy(dtype=float)

    if use_acc_gate and "acc_gate" in df_agg:
        acc  = df_agg["acc_gate"].to_numpy(dtype=float)
        gate = np.maximum(gyro, alpha * acc)
        source = f"max(gyro_gate, {alpha:.2f} × acc_gate)"
    else:
        gate   = gyro
        source = "gyro_gate"

    log_info = {
        "n_sensors_used": int(df["sensor"].nunique()) if "sensor" in df else 1,
        "gate_median": float(np.median(gate)),
        "gate_95p": float(np.percentile(gate, 95)),
    }

    return t, gate, source, "median_over_sensors", log_info


# ─── Build gate from EMG (fallback) ────────────────────────────────
def build_gate_from_emg(
    df_emg_trial: pd.DataFrame,
    time_bin_res: float = 0.001,
) -> tuple:
    df = df_emg_trial.copy()
    
    env_cols = [col for col in df.columns if col.startswith('env_norm_')]
    if not env_cols:
        env_cols = [col for col in df.columns if 'env_norm' in col.lower()]
    if not env_cols:
        raise ValueError("No env_norm columns found. Check column names.")
    
    print(f"  env_norm columns found for fallback: {env_cols}")
    
    # Important fix: use max instead of mean → more sensitive to strongest muscle activity
    df["emg_gate"] = df[env_cols].max(axis=1)
    
    df["t_bin"] = (df["t_emg"] / time_bin_res).round().astype(int)
    df_agg = df.groupby("t_bin", as_index=False).agg(
        {"t_emg": "mean", "emg_gate": "median"}
    ).sort_values("t_emg")
    
    t    = df_agg["t_emg"].to_numpy(dtype=float)
    gate = df_agg["emg_gate"].to_numpy(dtype=float)
    
    # If variance is very low, boost gate
    gate_var = np.var(gate)
    if gate_var < 1e-5:
        gate = gate * 5.0  # adjust factor as needed
        print("  Warning: gate variance too low – scaled gate")
    
    source = f"max_of_{len(env_cols)}_env_norm_muscles"
    
    log_info = {
        "n_muscles_used": len(env_cols),
        "gate_median": float(np.median(gate)),
        "gate_95p": float(np.percentile(gate, 95)),
        "gate_var": gate_var,
    }
    
    return t, gate, source, "median_over_time_bins", log_info


# ─── Plot gate and episodes ──────────────────────────────────────
def plot_trial_with_episodes(subject_name, trial_id, episodes_df):
    imu_path = PHASE2B_DIR / f"{subject_name}__imu_gate.parquet"
    emg_path = PHASE3_NORM_DIR / f"{subject_name}__emg_env_norm.parquet"  # fixed: correct path
    
    ep = episodes_df[
        (episodes_df["subject"] == subject_name) &
        (episodes_df["trial_id"] == trial_id) &
        (episodes_df["duration_s"] >= MIN_VALID_EPISODE_DURATION) &
        (episodes_df["peak_gate"] >= MIN_VALID_PEAK_GATE)  # show valid episodes only
    ].copy().sort_values("t_on")
    
    plt.figure(figsize=(14, 5))
    
    gate_source = "Unknown"
    
    if imu_path.exists():
        df_imu = pd.read_parquet(imu_path)
        df_trial = df_imu[(df_imu["subject"] == subject_name) & (df_imu["trial_id"] == trial_id)]
        if not df_trial.empty:
            df_trial = df_trial.sort_values("t_imu")
            t, gate, gate_source, _, _ = build_gate(df_trial, use_acc_gate=True, alpha=1.0)
            plt.plot(t, gate, lw=1.3, label=f"gate: {gate_source} (IMU)")
    
    if gate_source == "Unknown" and emg_path.exists():
        df_emg = pd.read_parquet(emg_path)
        df_trial = df_emg[(df_emg["subject"] == subject_name) & (df_emg["trial_id"] == trial_id)]
        if not df_trial.empty:
            df_trial = df_trial.sort_values("t_emg")
            t, gate, gate_source, _, _ = build_gate_from_emg(df_trial)
            plt.plot(t, gate, lw=1.3, label=f"gate: {gate_source} (EMG fallback)")
    
    if gate_source == "Unknown":
        print(f"No valid gate built for {trial_id} (neither IMU nor EMG)")
        plt.close()
        return
    
    for i, (_, r) in enumerate(ep.iterrows()):
        plt.axvspan(r["t_on"], r["t_off"], alpha=0.18, color='orange')
        plt.axvline(r["t_on"],  ls="--", lw=1.1, color='darkgreen',  label='on'  if i==0 else None)
        plt.axvline(r["t_off"], ls="--", lw=1.1, color='darkred',    label='off' if i==0 else None)
    
    plt.title(f"{subject_name} — {trial_id} — Episodes: {len(ep)}")
    plt.xlabel("Time (seconds)")
    plt.ylabel("Gate")
    plt.grid(True, alpha=0.35)
    plt.legend()
    plt.tight_layout()
    plt.show()

# ─── Main run for one subject ────────────────────────────────
def run_phase4_for_subject(
    subject_name: str,
    method: str = "mad",
    k_on: float = 5.0,
    k_off: float = 3.50,
    pad_s: float = 0.15,
    use_acc_gate: bool = True,
    alpha_acc: float = 1.0,
    compute_sanity_check: bool = True,
):
    imu_path = PHASE2B_DIR / f"{subject_name}__imu_gate.parquet"
    emg_path = PHASE3_NORM_DIR / f"{subject_name}__emg_env_norm.parquet"  # fixed: correct path
    
    # Load data
    df_imu = pd.DataFrame()
    df_emg = pd.DataFrame()
    
    if imu_path.exists():
        df_imu = pd.read_parquet(imu_path)
        df_imu["subject"] = df_imu["subject"].astype(str)
        df_imu["trial_id"] = df_imu["trial_id"].astype(str)
    
    if emg_path.exists():
        df_emg = pd.read_parquet(emg_path)
        df_emg["subject"] = df_emg["subject"].astype(str)
        df_emg["trial_id"] = df_emg["trial_id"].astype(str)
    
    if df_imu.empty and df_emg.empty:
        raise FileNotFoundError(f"No data found for {subject_name} (neither IMU nor EMG)")
    
    # Union of all trials
    trials_imu = set(df_imu["trial_id"].unique()) if not df_imu.empty else set()
    trials_emg = set(df_emg["trial_id"].unique()) if not df_emg.empty else set()
    all_trials = sorted(trials_imu | trials_emg)
    
    print(f"Total unique trials: {len(all_trials)}")
    print(f"  • With IMU:      {len(trials_imu)}")
    print(f"  • EMG only:      {len(trials_emg - trials_imu)}")
    
    events = []
    logs = []

    for trial_id in all_trials:
        use_imu = trial_id in trials_imu
        
        if use_imu:
            df_trial = df_imu[df_imu["trial_id"] == trial_id].copy()
            source_type = "IMU"
        else:
            df_trial = df_emg[df_emg["trial_id"] == trial_id].copy()
            source_type = "EMG_fallback"
        
        if df_trial.empty:
            print(f"Warning: trial {trial_id} is empty → skipped")
            continue
        
        print(f"Processing {trial_id:30} → source: {source_type}")
        
        task_key = next((k for k in TASK_SETTINGS if k in trial_id.lower()), "default")
        params = TASK_SETTINGS[task_key].copy()
        
        if not use_imu:
            params["min_duration_s"] *= EMG_FALLBACK_SETTINGS["min_duration_factor"]  # reduce min_duration for fallback
        
        if use_imu:
            t, gate, gate_source, _, gate_log = build_gate(
                df_trial, use_acc_gate=use_acc_gate, alpha=alpha_acc
            )
            curr_k_on  = k_on
            curr_k_off = k_off
            curr_peak_mult = 2.5
            curr_min_dur = params["min_duration_s"]
            curr_merge_gap = params["merge_gap_s"]
            curr_sec_merge = 0.8
            curr_pad_s = pad_s
            
            # general merging increase for oscillatory tasks (kept from before)
            if task_key in ["drinking", "lifting", "pick&place"]:
                curr_merge_gap = max(curr_merge_gap, 2.0)
                curr_sec_merge = max(curr_sec_merge, 4.0)
                print(f"  → Increased merging for task '{task_key}': merge_gap={curr_merge_gap}s, secondary={curr_sec_merge}s")
            
            # ─── targeted, stronger settings for drinking_exo ───────────────────────────────
            if "drinking" in trial_id.lower() and "exo" in trial_id.lower():
                curr_pad_s = 0.9          # much larger -> fuller coverage on both sides
                curr_k_off = 6.5          # higher -> harder offset, longer episodes
                curr_merge_gap = 4.5      # much stronger merging to join nearby intervals
                curr_sec_merge = 6.5      # strong secondary merge too
                curr_k_on = 3.0           # slightly lower for easier onset detection
                print(f"  → STRONG tuning for drinking_exo: pad={curr_pad_s}s | k_on={curr_k_on} | k_off={curr_k_off} | merge={curr_merge_gap}s | sec_merge={curr_sec_merge}s -> for full coverage")

            # ─── targeted, stronger settings for lifting_exo ───────────────────────────────
            if "lifting" in trial_id.lower() and "exo" in trial_id.lower():
                curr_pad_s = 0.04         # very small padding -> avoid extending into rest
                curr_k_off = 1.5          # very low -> very fast offset at dips
                curr_merge_gap = 0.4      # less merging -> finer segmentation
                curr_sec_merge = 0.8      # secondary merge limited too
                curr_k_on = 6.5           # slightly higher -> harder onset, avoids small false positives
                print(f"  → STRONG tuning for lifting_exo: pad={curr_pad_s}s | k_on={curr_k_on} | k_off={curr_k_off} | merge={curr_merge_gap}s | sec_merge={curr_sec_merge}s -> to shorten the episode and drop rest")

            gate_log["data_source"] = "imu"

        else:
            t, gate, gate_source, _, gate_log = build_gate_from_emg(df_trial)
            curr_k_on  = EMG_FALLBACK_SETTINGS["k_on"]
            curr_k_off = EMG_FALLBACK_SETTINGS["k_off"]
            curr_peak_mult = EMG_FALLBACK_SETTINGS["peak_multiplier"]
            curr_min_dur = params["min_duration_s"] * EMG_FALLBACK_SETTINGS["min_duration_factor"]
            curr_merge_gap = params["merge_gap_s"] * EMG_FALLBACK_SETTINGS["merge_gap_factor"]
            curr_sec_merge = EMG_FALLBACK_SETTINGS["secondary_merge_gap"]
            gate_log["data_source"] = "emg_only"

        # Gate quality check
        if compute_sanity_check:
            gate_var = np.var(gate)
            gate_log["gate_var"] = gate_var
            if gate_var < 1e-5:
                gate_log["sanity_flag"] = "very_low_variance"
                print(f"  → Warning: gate variance very low ({gate_var:.8f})")

        ep_df, ep_log = detect_episodes_hysteresis(
            t, gate,
            method=method,
            k_on=curr_k_on,
            k_off=curr_k_off,
            min_duration_s=curr_min_dur,
            merge_gap_s=curr_merge_gap,
            pad_s=pad_s,
            peak_multiplier=curr_peak_mult,
            secondary_merge_gap=curr_sec_merge,
        )

        # Filter out unrealistic / short / weak episodes (post-processing only)
        if not ep_df.empty:
            before_count = len(ep_df)
            
            # rule 1: sufficient duration
            ep_df = ep_df[ep_df["duration_s"] >= MIN_VALID_EPISODE_DURATION].copy()
            
            # rule 2: sufficient intensity (high peak_gate)
            ep_df = ep_df[ep_df["peak_gate"] >= MIN_VALID_PEAK_GATE].copy()

            # ─── Block saving for known bad trials ────────────────────────────────
            bad_trials = {"pick&place_high_exo", "pick&place_high_noexo"}
            if any(bad in trial_id.lower() for bad in bad_trials):
                print(f"  → Blocking bad trial: {trial_id} (known corrupted signal)")
                ep_df = pd.DataFrame()   # empty -> nothing is added to events
                ep_log["blocked_reason"] = "known_corrupted_signal"
                ep_log["warning"] = ep_log.get("warning", "") + "; blocked_bad_trial"
            
            discarded = before_count - len(ep_df)
            if discarded > 0:
                print(f"  → Discarded {discarded} episodes (short or low peak gate) in {trial_id}")
                ep_log["discarded_episodes"] = discarded
                ep_log["min_valid_duration"] = MIN_VALID_EPISODE_DURATION
                ep_log["min_valid_peak_gate"] = MIN_VALID_PEAK_GATE
        
        log = {
            "subject": subject_name,
            "trial_id": trial_id,
            "task_type": task_key,
            "gate_source": gate_source,
            "data_source": source_type,
            "method": method,
            **gate_log,
            **ep_log,
        }
        
        if ep_df.empty:
            print(f"  → Warning: no episodes detected for {trial_id} – log: {ep_log.get('warning', 'unknown')}")
            logs.append(log)
            continue

        # Calculate peak_gate for remaining episodes
        ep_df["peak_gate"] = [
            float(np.nanmax(gate[(t >= a) & (t <= b)])) if np.any((t >= a) & (t <= b)) else np.nan
            for a, b in zip(ep_df["t_on"], ep_df["t_off"])
        ]
        
        ep_df = ep_df.sort_values("t_on").reset_index(drop=True)
        ep_df["episode_id"] = range(1, len(ep_df) + 1)
        
        for _, row in ep_df.iterrows():
            r = {
                "subject": subject_name,
                "trial_id": trial_id,
                "episode_id": int(row["episode_id"]),
                "t_on": float(row["t_on"]),
                "t_off": float(row["t_off"]),
                "duration_s": float(row["duration_s"]),
                "peak_gate": row["peak_gate"],
                "thr_on": float(row["thr_on"]),
                "thr_off": float(row["thr_off"]),
                "gate_source": gate_source,
                "task_type": task_key,
                "data_source": source_type,
            }
            events.append(r)
        
        logs.append(log)
    
    events_df = pd.DataFrame(events)
    logs_df   = pd.DataFrame(logs)
    
    # Save
    events_df.to_parquet(PHASE4_DIR / f"{subject_name}__episodes.parquet", index=False)
    logs_df.to_csv(PHASE4_DIR / f"{subject_name}__episodes_log.csv", index=False)
    
    print(f"\nProcessing {subject_name} completed.")
    print(f"   → episodes: {PHASE4_DIR / f'{subject_name}__episodes.parquet'}")
    print(f"   → log:      {PHASE4_DIR / f'{subject_name}__episodes_log.csv'}")
    
    return events_df, logs_df

# ─── Run cell ────────────────────────────────────────────────
SUBJECT_NAME = "ALS_Subject_14"   # <- change only this line

print(f"\n=== Starting Phase 4 processing for {SUBJECT_NAME} ===\n")
print(f"Current filter settings: min_duration={MIN_VALID_EPISODE_DURATION}s | min_peak_gate={MIN_VALID_PEAK_GATE}")

events_df, logs_df = run_phase4_for_subject(
    subject_name = SUBJECT_NAME,
    method       = "mad",
    k_on         = 5.0,
    k_off        = 3.50,
    pad_s        = 0.15,
    use_acc_gate = True,
    alpha_acc    = 1.0,
    compute_sanity_check = True,
)

# ─── Display summary results ────────────────────────────────
if not events_df.empty:
    print("\nEpisode duration distribution (seconds):")
    display(events_df["duration_s"].describe())
    
    print("\nNumber of episodes per trial and data source:")
    display(events_df.groupby(["trial_id", "data_source"])["episode_id"].count().unstack(fill_value=0))
    
    # new: summary of discarded episodes from logs_df (if the filter fired)
    if 'discarded_episodes' in logs_df.columns:
        total_discarded = logs_df["discarded_episodes"].sum()
        print(f"\nTotal discarded short/weak episodes across all trials: {total_discarded}")
        display(logs_df[logs_df["discarded_episodes"] > 0][["trial_id", "discarded_episodes", "task_type"]])
    
    print(f"\nPlotting {len(events_df['trial_id'].unique())} valid trials (after filtering):")
    for tid in sorted(events_df["trial_id"].unique()):
        print(f"  → {tid}")
        plot_trial_with_episodes(SUBJECT_NAME, tid, events_df)
else:
    print("No episodes detected after filtering → check logs.")
    display(logs_df)

In [ ]:
# =========================================
# Phase 4 - Activity Detection (Onset / Offset) - Enhanced version for weak/ALS cases
# Notebook version (single file / single place)
# Last update: February 2026 + FIXES:
#   ✅ Plot ALL trials (even if 0 episodes)
#   ✅ Added plot_all_trials()
#   ✅ Optional: safe subject+trial filtering (won’t hurt even if file has one subject)
#   ✅ Two-stage placeholder still optional (kept as-is)
# =========================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import medfilt

# ─── Paths ────────────────────────────────────────
NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed"

PHASE2_DIR   = PROCESSED_ROOT / "phase_02_preprocess_emg"
PHASE2B_DIR  = PROCESSED_ROOT / "phase_02b_preprocess_imu"
PHASE3_DIR   = PROCESSED_ROOT / "phase_03_normalization"
PHASE3_NORM_DIR = PHASE3_DIR / "normalized_env_per_subject"
PHASE4_DIR   = PROCESSED_ROOT / "phase_04_events"

PHASE4_DIR.mkdir(parents=True, exist_ok=True)

print("Paths and initial settings loaded.")

# ─── Task-specific settings ─────────────────────────
TASK_SETTINGS = {
    "lifting":    {"min_duration_s": 1.8, "merge_gap_s": 0.35, "min_peak_gate": 100.0},
    "drinking":   {"min_duration_s": 1.2, "merge_gap_s": 0.30, "min_peak_gate": 60.0},
    "pick&place": {"min_duration_s": 2.2, "merge_gap_s": 1.20, "min_peak_gate": 75.0},
    "default":    {"min_duration_s": 1.5, "merge_gap_s": 0.40, "min_peak_gate": 70.0}
}

# Settings for weak/ALS fallback mode
WEAK_SIGNAL_SETTINGS = {
    "gate_95p_threshold": 50.0,     # gate_95p below this -> treated as weak
    "k_on": 2.8,
    "k_off": 1.5,
    "peak_multiplier": 1.4,
    "min_duration_factor": 0.65,
    "merge_gap_factor": 2.2,
    "secondary_merge_gap": 5.5,
    "alpha_acc_boost": 1.8,         # boost accelerometer weight in weak gates
    "smooth_kernel": 7,             # median-filter kernel for the gate
    "min_valid_duration_als": 2.0,
    "min_valid_peak_als": 35.0
}

# ─── Episode detection with hysteresis and post-processing ────────
def detect_episodes_hysteresis(
    t: np.ndarray,
    gate: np.ndarray,
    method: str = "mad",
    k_on: float = 5.0,
    k_off: float = 3.5,
    min_duration_s: float = 1.8,
    merge_gap_s: float = 0.45,
    pad_s: float = 0.15,
    peak_multiplier: float = 2.5,
    secondary_merge_gap: float = 0.8,
) -> tuple[pd.DataFrame, dict]:

    t    = np.asarray(t, dtype=float)
    gate = np.asarray(gate, dtype=float)

    mask_valid = np.isfinite(t) & np.isfinite(gate)
    t    = t[mask_valid]
    gate = gate[mask_valid]

    log = {
        "raw_samples": len(t),
        "gate_median": float(np.median(gate)) if len(gate) > 0 else np.nan,
        "gate_95p": float(np.percentile(gate, 95)) if len(gate) > 0 else np.nan,
    }

    if len(t) < 20:
        log["warning"] = "too_few_samples"
        return pd.DataFrame(columns=["t_on","t_off","duration_s","thr_on","thr_off","peak_gate"]), log

    med = float(np.median(gate))
    mad = float(np.median(np.abs(gate - med))) + 1e-12
    thr_on  = med + k_on  * mad
    thr_off = med + k_off * mad

    log["thr_on"]  = float(thr_on)
    log["thr_off"] = float(thr_off)

    # Auto-adjust if threshold too high
    if thr_on > log["gate_95p"] * 1.1:
        k_on = max(1.2, k_on * 0.65)
        thr_on = med + k_on * mad
        # (Optional) keep thr_off consistent:
        thr_off = med + k_off * mad
        log["thr_on_adjusted"] = float(thr_on)
        log["warning"] = "thr_adjusted_down"

    episodes = []
    in_episode = False
    t_start = None

    for i in range(len(t)):
        if not in_episode:
            if gate[i] >= thr_on:
                in_episode = True
                t_start = float(t[i])
        else:
            if gate[i] <= thr_off:
                episodes.append([t_start, float(t[i])])
                in_episode = False
                t_start = None

    if in_episode:
        episodes.append([t_start, float(t[-1])])
        log["last_episode_open"] = True

    if not episodes:
        log["warning"] = "no_episodes_detected"
        return pd.DataFrame(columns=["t_on","t_off","duration_s","thr_on","thr_off","peak_gate"]), log

    t_min, t_max = float(t[0]), float(t[-1])
    padded = [[max(t_min, a - pad_s), min(t_max, b + pad_s)] for a, b in episodes]

    padded.sort(key=lambda x: x[0])
    merged = [padded[0]]
    for curr in padded[1:]:
        if curr[0] - merged[-1][1] <= merge_gap_s:
            merged[-1][1] = max(merged[-1][1], curr[1])
        else:
            merged.append(curr)

    final_merged = []
    current = merged[0]
    for nxt in merged[1:]:
        if nxt[0] - current[1] <= secondary_merge_gap:
            current[1] = max(current[1], nxt[1])
        else:
            final_merged.append(current)
            current = nxt
    final_merged.append(current)

    adaptive_peak_thr = med + peak_multiplier * mad
    log["adaptive_peak_thr"] = float(adaptive_peak_thr)

    final_ep = []
    for a, b in final_merged:
        dur = b - a
        if dur < min_duration_s:
            continue

        segment_gate = gate[(t >= a) & (t <= b)]
        if len(segment_gate) == 0:
            continue
        peak = float(np.max(segment_gate))

        if peak >= adaptive_peak_thr:
            final_ep.append([a, b, dur, thr_on, thr_off, peak])

    df_ep = pd.DataFrame(final_ep, columns=["t_on", "t_off", "duration_s", "thr_on", "thr_off", "peak_gate"])
    log["n_episodes_after_postproc"] = int(len(df_ep))

    return df_ep, log


# ─── Build gate from IMU ────────────────────────────────────────
def build_gate(
    df_imu_trial: pd.DataFrame,
    use_acc_gate: bool = True,
    alpha: float = 1.0,
    time_bin_res: float = 0.001,
    smooth_kernel: int = 5,
    is_weak: bool = False,
) -> tuple:
    df = df_imu_trial.copy()

    df["t_bin"] = (df["t_imu"] / time_bin_res).round().astype(int)

    agg_dict = {"t_imu": "mean", "gyro_gate": "median"}
    if use_acc_gate and "acc_gate" in df.columns:
        agg_dict["acc_gate"] = "median"

    df_agg = df.groupby("t_bin", as_index=False).agg(agg_dict).sort_values("t_imu")

    t    = df_agg["t_imu"].to_numpy(dtype=float)
    gyro = df_agg["gyro_gate"].to_numpy(dtype=float)

    if use_acc_gate and ("acc_gate" in df_agg.columns):
        acc  = df_agg["acc_gate"].to_numpy(dtype=float)
        if is_weak:
            alpha = max(alpha, WEAK_SIGNAL_SETTINGS["alpha_acc_boost"])
        gate = np.maximum(gyro, alpha * acc)
        source = f"max(gyro_gate, {alpha:.2f} × acc_gate)"
    else:
        gate   = gyro
        source = "gyro_gate"

    # Smoothing for fragmented/weak signals
    if smooth_kernel > 1:
        # ensure odd kernel for medfilt
        if smooth_kernel % 2 == 0:
            smooth_kernel += 1
        gate = medfilt(gate, kernel_size=smooth_kernel)

    log_info = {
        "n_sensors_used": int(df["sensor"].nunique()) if "sensor" in df.columns else 1,
        "gate_median": float(np.median(gate)) if len(gate) else np.nan,
        "gate_95p": float(np.percentile(gate, 95)) if len(gate) else np.nan,
        "smooth_kernel": int(smooth_kernel),
    }

    return t, gate, source, "median_over_sensors", log_info


# ─── Build gate from EMG (fallback) ────────────────────────────────
def build_gate_from_emg(
    df_emg_trial: pd.DataFrame,
    time_bin_res: float = 0.001,
    smooth_kernel: int = 5,
    apply_tkeo: bool = False,
) -> tuple:
    df = df_emg_trial.copy()

    env_cols = [col for col in df.columns if col.startswith('env_norm_')]
    if not env_cols:
        env_cols = [col for col in df.columns if 'env_norm' in col.lower()]
    if not env_cols:
        raise ValueError("No env_norm columns found. Check column names.")

    df["emg_gate"] = df[env_cols].max(axis=1)

    df["t_bin"] = (df["t_emg"] / time_bin_res).round().astype(int)
    df_agg = df.groupby("t_bin", as_index=False).agg(
        {"t_emg": "mean", "emg_gate": "median"}
    ).sort_values("t_emg")

    t    = df_agg["t_emg"].to_numpy(dtype=float)
    gate = df_agg["emg_gate"].to_numpy(dtype=float)

    # Smoothing
    if smooth_kernel > 1:
        if smooth_kernel % 2 == 0:
            smooth_kernel += 1
        gate = medfilt(gate, kernel_size=smooth_kernel)

    # Optional TKEO for weak bursts
    if apply_tkeo:
        if len(gate) > 2:
            tkeo_vals = gate[1:-1]**2 - gate[:-2] * gate[2:]
            gate = np.concatenate(([gate[0]], tkeo_vals, [gate[-1]]))

    gate_var = float(np.var(gate)) if len(gate) else np.nan
    if np.isfinite(gate_var) and gate_var < 1e-5:
        p95 = np.percentile(gate, 95) if len(gate) else 0.0
        scale_factor = (p95 / 25.0) if p95 > 0 else 4.0
        gate *= scale_factor

    source = f"max_of_{len(env_cols)}_env_norm_muscles"

    log_info = {
        "n_muscles_used": int(len(env_cols)),
        "gate_median": float(np.median(gate)) if len(gate) else np.nan,
        "gate_95p": float(np.percentile(gate, 95)) if len(gate) else np.nan,
        "gate_var": float(np.var(gate)) if len(gate) else np.nan,
    }

    return t, gate, source, "median_over_time_bins", log_info


# ─── Plot: ALWAYS plot gate, with or without episodes ───────────────────────────
def plot_trial_with_episodes(subject_name, trial_id, events_df, logs_df=None):
    """
    Always plot gate for a specific trial.
    If episodes exist → draw spans/lines.
    If no episodes → still plot gate + show message.
    Optionally uses logs_df to decide is_weak when no episodes exist.
    """
    imu_path = PHASE2B_DIR / f"{subject_name}__imu_gate.parquet"
    emg_path = PHASE3_NORM_DIR / f"{subject_name}__emg_env_norm.parquet"

    ep = events_df[
        (events_df["subject"] == subject_name) &
        (events_df["trial_id"] == trial_id)
    ].copy()

    ep = ep.sort_values("t_on") if not ep.empty else ep

    # decide is_weak for plotting
    if (not ep.empty) and ("is_weak" in ep.columns):
        is_weak = bool(ep["is_weak"].iloc[0])
    elif logs_df is not None:
        rr = logs_df[(logs_df["subject"] == subject_name) & (logs_df["trial_id"] == trial_id)]
        is_weak = bool(rr["is_weak_adapted"].iloc[0]) if (not rr.empty and "is_weak_adapted" in rr.columns) else False
    else:
        is_weak = False

    plt.figure(figsize=(14, 6))

    gate_source = "Unknown"
    gate_plotted = False

    # IMU first
    if imu_path.exists():
        df_imu = pd.read_parquet(imu_path)
        # safe filter (even if single-subject file)
        if "subject" in df_imu.columns:
            df_trial = df_imu[(df_imu["subject"].astype(str) == str(subject_name)) & (df_imu["trial_id"].astype(str) == str(trial_id))]
        else:
            df_trial = df_imu[(df_imu["trial_id"].astype(str) == str(trial_id))]
        if not df_trial.empty:
            df_trial = df_trial.sort_values("t_imu")

            alpha = WEAK_SIGNAL_SETTINGS["alpha_acc_boost"] if is_weak else 1.0
            smooth_k = WEAK_SIGNAL_SETTINGS["smooth_kernel"] if is_weak else 5

            t, gate, gate_source, _, _ = build_gate(
                df_trial,
                use_acc_gate=True,
                alpha=alpha,
                smooth_kernel=smooth_k,
                is_weak=is_weak
            )
            plt.plot(t, gate, lw=1.4, label=f"gate: {gate_source} (IMU)")
            gate_plotted = True

    # EMG fallback
    if (not gate_plotted) and emg_path.exists():
        df_emg = pd.read_parquet(emg_path)
        if "subject" in df_emg.columns:
            df_trial = df_emg[(df_emg["subject"].astype(str) == str(subject_name)) & (df_emg["trial_id"].astype(str) == str(trial_id))]
        else:
            df_trial = df_emg[(df_emg["trial_id"].astype(str) == str(trial_id))]
        if not df_trial.empty:
            df_trial = df_trial.sort_values("t_emg")

            smooth_k = WEAK_SIGNAL_SETTINGS["smooth_kernel"] if is_weak else 5
            apply_tkeo = is_weak

            t, gate, gate_source, _, _ = build_gate_from_emg(
                df_trial,
                smooth_kernel=smooth_k,
                apply_tkeo=apply_tkeo
            )
            plt.plot(t, gate, lw=1.4, label=f"gate: {gate_source} (EMG fallback)")
            gate_plotted = True

    if not gate_plotted:
        print(f"Warning: No valid gate could be built for {trial_id} (neither IMU nor EMG)")
        plt.close()
        return

    # Draw episodes if exist
    if not ep.empty:
        for i, (_, row) in enumerate(ep.iterrows()):
            plt.axvspan(row["t_on"], row["t_off"], alpha=0.15, color='orange', lw=0)
            plt.axvline(row["t_on"], ls="--", lw=1.4, color='green', label='on' if i == 0 else None)
            plt.axvline(row["t_off"], ls="--", lw=1.4, color='red',   label='off' if i == 0 else None)
        subtitle = f"Episodes: {len(ep)}"
    else:
        subtitle = "Episodes: 0 (none detected)"

    title_suffix = " (weak adapted)" if is_weak else ""
    plt.title(f"{subject_name} — {trial_id} — {subtitle}{title_suffix}", fontsize=13)
    plt.xlabel("Time (seconds)")
    plt.ylabel("Gate value")
    plt.grid(True, alpha=0.3, linestyle='--')
    plt.legend(loc='upper right', fontsize=9)
    plt.tight_layout()
    plt.show()


def plot_all_trials(subject_name, events_df, logs_df=None, sort=True):
    """
    Plot all trials for a subject.
    Uses logs_df to get the complete trial list (recommended), otherwise falls back to events_df.
    """
    if logs_df is not None and ("trial_id" in logs_df.columns):
        trial_ids = logs_df[logs_df["subject"].astype(str) == str(subject_name)]["trial_id"].astype(str).unique().tolist()
    else:
        trial_ids = events_df[events_df["subject"].astype(str) == str(subject_name)]["trial_id"].astype(str).unique().tolist()

    if sort:
        trial_ids = sorted(trial_ids)

    print(f"Plotting {len(trial_ids)} trials for {subject_name} ...")

    for trial_id in trial_ids:
        plot_trial_with_episodes(subject_name, trial_id, events_df, logs_df=logs_df)


# ─── Main run for one subject ────────────────────────────────
def run_phase4_for_subject(
    subject_name: str,
    method: str = "mad",
    k_on: float = 5.0,
    k_off: float = 3.50,
    pad_s: float = 0.15,
    use_acc_gate: bool = True,
    alpha_acc: float = 1.0,
    compute_sanity_check: bool = True,
    enable_weak_adaptation: bool = True,
    enable_two_stage_refine: bool = True,
):
    imu_path = PHASE2B_DIR / f"{subject_name}__imu_gate.parquet"
    emg_path = PHASE3_NORM_DIR / f"{subject_name}__emg_env_norm.parquet"

    df_imu = pd.DataFrame()
    df_emg = pd.DataFrame()

    if imu_path.exists():
        df_imu = pd.read_parquet(imu_path)
        if "subject" in df_imu.columns:
            df_imu["subject"] = df_imu["subject"].astype(str)
        df_imu["trial_id"] = df_imu["trial_id"].astype(str)

    if emg_path.exists():
        df_emg = pd.read_parquet(emg_path)
        if "subject" in df_emg.columns:
            df_emg["subject"] = df_emg["subject"].astype(str)
        df_emg["trial_id"] = df_emg["trial_id"].astype(str)

    if df_imu.empty and df_emg.empty:
        raise FileNotFoundError(f"No data found for {subject_name}")

    trials_imu = set(df_imu["trial_id"].unique()) if not df_imu.empty else set()
    trials_emg = set(df_emg["trial_id"].unique()) if not df_emg.empty else set()
    all_trials = sorted(trials_imu | trials_emg)

    print(f"Total unique trials: {len(all_trials)}")

    events = []
    logs = []

    is_als_subject = "ALS" in subject_name.upper()

    for trial_id in all_trials:
        use_imu = trial_id in trials_imu

        if use_imu:
            # safe filter (works even if single-subject file)
            if ("subject" in df_imu.columns):
                df_trial = df_imu[(df_imu["subject"] == str(subject_name)) & (df_imu["trial_id"] == str(trial_id))].copy()
            else:
                df_trial = df_imu[(df_imu["trial_id"] == str(trial_id))].copy()
            source_type = "IMU"
        else:
            if ("subject" in df_emg.columns):
                df_trial = df_emg[(df_emg["subject"] == str(subject_name)) & (df_emg["trial_id"] == str(trial_id))].copy()
            else:
                df_trial = df_emg[(df_emg["trial_id"] == str(trial_id))].copy()
            source_type = "EMG_fallback"

        if df_trial.empty:
            continue

        print(f"Processing {trial_id:30} → source: {source_type}")

        task_key = next((k for k in TASK_SETTINGS if k in trial_id.lower()), "default")
        params = TASK_SETTINGS[task_key].copy()

        # Default params
        curr_k_on  = k_on
        curr_k_off = k_off
        curr_peak_mult = 2.5
        curr_min_dur = params["min_duration_s"]
        curr_merge_gap = params["merge_gap_s"]
        curr_sec_merge = 0.8
        curr_alpha = alpha_acc
        smooth_k = 5
        apply_tkeo = False
        is_weak = False

        # Weak/ALS adaptation
        if enable_weak_adaptation:
            if use_imu:
                _, gate_temp, _, _, gate_log_temp = build_gate(
                    df_trial, use_acc_gate=use_acc_gate, alpha=curr_alpha,
                    smooth_kernel=1
                )
            else:
                _, gate_temp, _, _, gate_log_temp = build_gate_from_emg(df_trial, smooth_kernel=1)

            gate_95p = float(gate_log_temp.get("gate_95p", 100.0))

            if is_als_subject or (gate_95p < WEAK_SIGNAL_SETTINGS["gate_95p_threshold"]):
                is_weak = True
                print(f"  → Weak/ALS signal detected (95p={gate_95p:.1f}) → adapting")

                curr_k_on  = WEAK_SIGNAL_SETTINGS["k_on"]
                curr_k_off = WEAK_SIGNAL_SETTINGS["k_off"]
                curr_peak_mult = WEAK_SIGNAL_SETTINGS["peak_multiplier"]
                curr_min_dur *= WEAK_SIGNAL_SETTINGS["min_duration_factor"]
                curr_merge_gap *= WEAK_SIGNAL_SETTINGS["merge_gap_factor"]
                curr_sec_merge = WEAK_SIGNAL_SETTINGS["secondary_merge_gap"]
                smooth_k = WEAK_SIGNAL_SETTINGS["smooth_kernel"]
                apply_tkeo = not use_imu

                if use_imu:
                    curr_alpha = WEAK_SIGNAL_SETTINGS["alpha_acc_boost"]

        # Build final gate
        if use_imu:
            t, gate, gate_source, _, gate_log = build_gate(
                df_trial,
                use_acc_gate=use_acc_gate,
                alpha=curr_alpha,
                smooth_kernel=smooth_k,
                is_weak=is_weak
            )
            gate_log["data_source"] = "imu"
        else:
            t, gate, gate_source, _, gate_log = build_gate_from_emg(
                df_trial,
                smooth_kernel=smooth_k,
                apply_tkeo=apply_tkeo
            )
            gate_log["data_source"] = "emg_only"

        # Gate quality check
        if compute_sanity_check:
            gate_var = float(np.var(gate)) if len(gate) else np.nan
            gate_log["gate_var"] = gate_var
            if np.isfinite(gate_var) and gate_var < 1e-5:
                gate_log["sanity_flag"] = "very_low_variance"

        ep_df, ep_log = detect_episodes_hysteresis(
            t, gate,
            method=method,
            k_on=curr_k_on,
            k_off=curr_k_off,
            min_duration_s=curr_min_dur,
            merge_gap_s=curr_merge_gap,
            pad_s=pad_s,
            peak_multiplier=curr_peak_mult,
            secondary_merge_gap=curr_sec_merge,
        )

        # Post-filtering
        min_dur = WEAK_SIGNAL_SETTINGS["min_valid_duration_als"] if is_weak else 3.5
        min_peak = WEAK_SIGNAL_SETTINGS["min_valid_peak_als"] if is_weak else 50.0

        if not ep_df.empty:
            before = len(ep_df)
            ep_df = ep_df[
                (ep_df["duration_s"] >= min_dur) &
                (ep_df["peak_gate"] >= min_peak)
            ].copy()
            discarded = before - len(ep_df)
            if discarded > 0:
                print(f"  → Discarded {discarded} episodes")
                ep_log["discarded_episodes"] = int(discarded)

        # Optional: Two-stage refine (kept as placeholder behavior)
        if enable_two_stage_refine and len(ep_df) <= 1 and is_weak:
            print("  → Two-stage refine activated (placeholder)")

        log = {
            "subject": subject_name,
            "trial_id": trial_id,
            "task_type": task_key,
            "gate_source": gate_source,
            "data_source": source_type,
            "method": method,
            "is_weak_adapted": bool(is_weak),
            **gate_log,
            **ep_log,
        }

        if ep_df.empty:
            logs.append(log)
            continue

        # recompute peak_gate precisely on final gate
        ep_df["peak_gate"] = [
            float(np.nanmax(gate[(t >= a) & (t <= b)])) if np.any((t >= a) & (t <= b)) else np.nan
            for a, b in zip(ep_df["t_on"], ep_df["t_off"])
        ]

        ep_df = ep_df.sort_values("t_on").reset_index(drop=True)
        ep_df["episode_id"] = range(1, len(ep_df) + 1)

        for _, row in ep_df.iterrows():
            events.append({
                "subject": subject_name,
                "trial_id": trial_id,
                "episode_id": int(row["episode_id"]),
                "t_on": float(row["t_on"]),
                "t_off": float(row["t_off"]),
                "duration_s": float(row["duration_s"]),
                "peak_gate": float(row["peak_gate"]) if pd.notna(row["peak_gate"]) else np.nan,
                "thr_on": float(row["thr_on"]),
                "thr_off": float(row["thr_off"]),
                "gate_source": gate_source,
                "task_type": task_key,
                "data_source": source_type,
                "is_weak": bool(is_weak),
            })

        logs.append(log)

    events_df = pd.DataFrame(events)
    logs_df   = pd.DataFrame(logs)

    events_df.to_parquet(PHASE4_DIR / f"{subject_name}__episodes.parquet", index=False)
    logs_df.to_csv(PHASE4_DIR / f"{subject_name}__episodes_log.csv", index=False)

    print(f"\nProcessing {subject_name} completed.")
    return events_df, logs_df


# ─── Example usage ───────────────────────────────────────────────
events_df, logs_df = run_phase4_for_subject("ALS_Subject_7", enable_weak_adaptation=True)

# ✅ NEW: Plot every trial (even if no episodes were detected)
plot_all_trials("ALS_Subject_7", events_df, logs_df)


In [ ]:
# =========================================
# Phase 4 - Activity Detection (Onset / Offset) - Enhanced version for weak/ALS cases
# Notebook version (single file / single place)
# Last update: February 2026 + FIXES:
#   ✅ Plot ALL trials (even if 0 episodes)
#   ✅ Added plot_all_trials()
#   ✅ Optional: safe subject+trial filtering (won’t hurt even if file has one subject)
#   ✅ Two-stage placeholder still optional (kept as-is)
# =========================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import medfilt

# ─── Paths ────────────────────────────────────────
NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed"

PHASE2_DIR   = PROCESSED_ROOT / "phase_02_preprocess_emg"
PHASE2B_DIR  = PROCESSED_ROOT / "phase_02b_preprocess_imu"
PHASE3_DIR   = PROCESSED_ROOT / "phase_03_normalization"
PHASE3_NORM_DIR = PHASE3_DIR / "normalized_env_per_subject"
PHASE4_DIR   = PROCESSED_ROOT / "phase_04_events"

PHASE4_DIR.mkdir(parents=True, exist_ok=True)

print("Paths and initial settings loaded.")

# ─── Task-specific settings ─────────────────────────
TASK_SETTINGS = {
    "lifting":    {"min_duration_s": 1.8, "merge_gap_s": 0.35, "min_peak_gate": 100.0},
    "drinking":   {"min_duration_s": 1.2, "merge_gap_s": 0.30, "min_peak_gate": 60.0},
    "pick&place": {"min_duration_s": 2.2, "merge_gap_s": 1.20, "min_peak_gate": 75.0},
    "default":    {"min_duration_s": 1.5, "merge_gap_s": 0.40, "min_peak_gate": 70.0}
}

# Settings for weak/ALS fallback mode
WEAK_SIGNAL_SETTINGS = {
    "gate_95p_threshold": 50.0,     # gate_95p below this -> treated as weak
    "k_on": 2.8,
    "k_off": 1.5,
    "peak_multiplier": 1.4,
    "min_duration_factor": 0.65,
    "merge_gap_factor": 2.2,
    "secondary_merge_gap": 5.5,
    "alpha_acc_boost": 1.8,         # boost accelerometer weight in weak gates
    "smooth_kernel": 7,             # median-filter kernel for the gate
    "min_valid_duration_als": 2.0,
    "min_valid_peak_als": 35.0
}

# ─── Episode detection with hysteresis and post-processing ────────
def detect_episodes_hysteresis(
    t: np.ndarray,
    gate: np.ndarray,
    method: str = "mad",
    k_on: float = 5.0,
    k_off: float = 3.5,
    min_duration_s: float = 1.8,
    merge_gap_s: float = 0.45,
    pad_s: float = 0.15,
    peak_multiplier: float = 2.5,
    secondary_merge_gap: float = 0.8,
) -> tuple[pd.DataFrame, dict]:

    t    = np.asarray(t, dtype=float)
    gate = np.asarray(gate, dtype=float)

    mask_valid = np.isfinite(t) & np.isfinite(gate)
    t    = t[mask_valid]
    gate = gate[mask_valid]

    log = {
        "raw_samples": len(t),
        "gate_median": float(np.median(gate)) if len(gate) > 0 else np.nan,
        "gate_95p": float(np.percentile(gate, 95)) if len(gate) > 0 else np.nan,
    }

    if len(t) < 20:
        log["warning"] = "too_few_samples"
        return pd.DataFrame(columns=["t_on","t_off","duration_s","thr_on","thr_off","peak_gate"]), log

    med = float(np.median(gate))
    mad = float(np.median(np.abs(gate - med))) + 1e-12
    thr_on  = med + k_on  * mad
    thr_off = med + k_off * mad

    log["thr_on"]  = float(thr_on)
    log["thr_off"] = float(thr_off)

    # Auto-adjust if threshold too high
    if thr_on > log["gate_95p"] * 1.1:
        k_on = max(1.2, k_on * 0.65)
        thr_on = med + k_on * mad
        # (Optional) keep thr_off consistent:
        thr_off = med + k_off * mad
        log["thr_on_adjusted"] = float(thr_on)
        log["warning"] = "thr_adjusted_down"

    episodes = []
    in_episode = False
    t_start = None

    for i in range(len(t)):
        if not in_episode:
            if gate[i] >= thr_on:
                in_episode = True
                t_start = float(t[i])
        else:
            if gate[i] <= thr_off:
                episodes.append([t_start, float(t[i])])
                in_episode = False
                t_start = None

    if in_episode:
        episodes.append([t_start, float(t[-1])])
        log["last_episode_open"] = True

    if not episodes:
        log["warning"] = "no_episodes_detected"
        return pd.DataFrame(columns=["t_on","t_off","duration_s","thr_on","thr_off","peak_gate"]), log

    t_min, t_max = float(t[0]), float(t[-1])
    padded = [[max(t_min, a - pad_s), min(t_max, b + pad_s)] for a, b in episodes]

    padded.sort(key=lambda x: x[0])
    merged = [padded[0]]
    for curr in padded[1:]:
        if curr[0] - merged[-1][1] <= merge_gap_s:
            merged[-1][1] = max(merged[-1][1], curr[1])
        else:
            merged.append(curr)

    final_merged = []
    current = merged[0]
    for nxt in merged[1:]:
        if nxt[0] - current[1] <= secondary_merge_gap:
            current[1] = max(current[1], nxt[1])
        else:
            final_merged.append(current)
            current = nxt
    final_merged.append(current)

    adaptive_peak_thr = med + peak_multiplier * mad
    log["adaptive_peak_thr"] = float(adaptive_peak_thr)

    final_ep = []
    for a, b in final_merged:
        dur = b - a
        if dur < min_duration_s:
            continue

        segment_gate = gate[(t >= a) & (t <= b)]
        if len(segment_gate) == 0:
            continue
        peak = float(np.max(segment_gate))

        if peak >= adaptive_peak_thr:
            final_ep.append([a, b, dur, thr_on, thr_off, peak])

    df_ep = pd.DataFrame(final_ep, columns=["t_on", "t_off", "duration_s", "thr_on", "thr_off", "peak_gate"])
    log["n_episodes_after_postproc"] = int(len(df_ep))

    return df_ep, log


# ─── Build gate from IMU ────────────────────────────────────────
def build_gate(
    df_imu_trial: pd.DataFrame,
    use_acc_gate: bool = True,
    alpha: float = 1.0,
    time_bin_res: float = 0.001,
    smooth_kernel: int = 5,
    is_weak: bool = False,
) -> tuple:
    df = df_imu_trial.copy()

    df["t_bin"] = (df["t_imu"] / time_bin_res).round().astype(int)

    agg_dict = {"t_imu": "mean", "gyro_gate": "median"}
    if use_acc_gate and "acc_gate" in df.columns:
        agg_dict["acc_gate"] = "median"

    df_agg = df.groupby("t_bin", as_index=False).agg(agg_dict).sort_values("t_imu")

    t    = df_agg["t_imu"].to_numpy(dtype=float)
    gyro = df_agg["gyro_gate"].to_numpy(dtype=float)

    if use_acc_gate and ("acc_gate" in df_agg.columns):
        acc  = df_agg["acc_gate"].to_numpy(dtype=float)
        if is_weak:
            alpha = max(alpha, WEAK_SIGNAL_SETTINGS["alpha_acc_boost"])
        gate = np.maximum(gyro, alpha * acc)
        source = f"max(gyro_gate, {alpha:.2f} × acc_gate)"
    else:
        gate   = gyro
        source = "gyro_gate"

    # Smoothing for fragmented/weak signals
    if smooth_kernel > 1:
        # ensure odd kernel for medfilt
        if smooth_kernel % 2 == 0:
            smooth_kernel += 1
        gate = medfilt(gate, kernel_size=smooth_kernel)

    log_info = {
        "n_sensors_used": int(df["sensor"].nunique()) if "sensor" in df.columns else 1,
        "gate_median": float(np.median(gate)) if len(gate) else np.nan,
        "gate_95p": float(np.percentile(gate, 95)) if len(gate) else np.nan,
        "smooth_kernel": int(smooth_kernel),
    }

    return t, gate, source, "median_over_sensors", log_info


# ─── Build gate from EMG (fallback) ────────────────────────────────
def build_gate_from_emg(
    df_emg_trial: pd.DataFrame,
    time_bin_res: float = 0.001,
    smooth_kernel: int = 5,
    apply_tkeo: bool = False,
) -> tuple:
    df = df_emg_trial.copy()

    env_cols = [col for col in df.columns if col.startswith('env_norm_')]
    if not env_cols:
        env_cols = [col for col in df.columns if 'env_norm' in col.lower()]
    if not env_cols:
        raise ValueError("No env_norm columns found. Check column names.")

    df["emg_gate"] = df[env_cols].max(axis=1)

    df["t_bin"] = (df["t_emg"] / time_bin_res).round().astype(int)
    df_agg = df.groupby("t_bin", as_index=False).agg(
        {"t_emg": "mean", "emg_gate": "median"}
    ).sort_values("t_emg")

    t    = df_agg["t_emg"].to_numpy(dtype=float)
    gate = df_agg["emg_gate"].to_numpy(dtype=float)

    # Smoothing
    if smooth_kernel > 1:
        if smooth_kernel % 2 == 0:
            smooth_kernel += 1
        gate = medfilt(gate, kernel_size=smooth_kernel)

    # Optional TKEO for weak bursts
    if apply_tkeo:
        if len(gate) > 2:
            tkeo_vals = gate[1:-1]**2 - gate[:-2] * gate[2:]
            gate = np.concatenate(([gate[0]], tkeo_vals, [gate[-1]]))

    gate_var = float(np.var(gate)) if len(gate) else np.nan
    if np.isfinite(gate_var) and gate_var < 1e-5:
        p95 = np.percentile(gate, 95) if len(gate) else 0.0
        scale_factor = (p95 / 25.0) if p95 > 0 else 4.0
        gate *= scale_factor

    source = f"max_of_{len(env_cols)}_env_norm_muscles"

    log_info = {
        "n_muscles_used": int(len(env_cols)),
        "gate_median": float(np.median(gate)) if len(gate) else np.nan,
        "gate_95p": float(np.percentile(gate, 95)) if len(gate) else np.nan,
        "gate_var": float(np.var(gate)) if len(gate) else np.nan,
    }

    return t, gate, source, "median_over_time_bins", log_info


# ─── Plot: ALWAYS plot gate, with or without episodes ───────────────────────────
def plot_trial_with_episodes(subject_name, trial_id, events_df, logs_df=None):
    """
    Always plot gate for a specific trial.
    If episodes exist → draw spans/lines.
    If no episodes → still plot gate + show message.
    Optionally uses logs_df to decide is_weak when no episodes exist.
    """
    imu_path = PHASE2B_DIR / f"{subject_name}__imu_gate.parquet"
    emg_path = PHASE3_NORM_DIR / f"{subject_name}__emg_env_norm.parquet"

    ep = events_df[
        (events_df["subject"] == subject_name) &
        (events_df["trial_id"] == trial_id)
    ].copy()

    ep = ep.sort_values("t_on") if not ep.empty else ep

    # decide is_weak for plotting
    if (not ep.empty) and ("is_weak" in ep.columns):
        is_weak = bool(ep["is_weak"].iloc[0])
    elif logs_df is not None:
        rr = logs_df[(logs_df["subject"] == subject_name) & (logs_df["trial_id"] == trial_id)]
        is_weak = bool(rr["is_weak_adapted"].iloc[0]) if (not rr.empty and "is_weak_adapted" in rr.columns) else False
    else:
        is_weak = False

    plt.figure(figsize=(14, 6))

    gate_source = "Unknown"
    gate_plotted = False

    # IMU first
    if imu_path.exists():
        df_imu = pd.read_parquet(imu_path)
        # safe filter (even if single-subject file)
        if "subject" in df_imu.columns:
            df_trial = df_imu[(df_imu["subject"].astype(str) == str(subject_name)) & (df_imu["trial_id"].astype(str) == str(trial_id))]
        else:
            df_trial = df_imu[(df_imu["trial_id"].astype(str) == str(trial_id))]
        if not df_trial.empty:
            df_trial = df_trial.sort_values("t_imu")

            alpha = WEAK_SIGNAL_SETTINGS["alpha_acc_boost"] if is_weak else 1.0
            smooth_k = WEAK_SIGNAL_SETTINGS["smooth_kernel"] if is_weak else 5

            t, gate, gate_source, _, _ = build_gate(
                df_trial,
                use_acc_gate=True,
                alpha=alpha,
                smooth_kernel=smooth_k,
                is_weak=is_weak
            )
            plt.plot(t, gate, lw=1.4, label=f"gate: {gate_source} (IMU)")
            gate_plotted = True

    # EMG fallback
    if (not gate_plotted) and emg_path.exists():
        df_emg = pd.read_parquet(emg_path)
        if "subject" in df_emg.columns:
            df_trial = df_emg[(df_emg["subject"].astype(str) == str(subject_name)) & (df_emg["trial_id"].astype(str) == str(trial_id))]
        else:
            df_trial = df_emg[(df_emg["trial_id"].astype(str) == str(trial_id))]
        if not df_trial.empty:
            df_trial = df_trial.sort_values("t_emg")

            smooth_k = WEAK_SIGNAL_SETTINGS["smooth_kernel"] if is_weak else 5
            apply_tkeo = is_weak

            t, gate, gate_source, _, _ = build_gate_from_emg(
                df_trial,
                smooth_kernel=smooth_k,
                apply_tkeo=apply_tkeo
            )
            plt.plot(t, gate, lw=1.4, label=f"gate: {gate_source} (EMG fallback)")
            gate_plotted = True

    if not gate_plotted:
        print(f"Warning: No valid gate could be built for {trial_id} (neither IMU nor EMG)")
        plt.close()
        return

    # Draw episodes if exist
    if not ep.empty:
        for i, (_, row) in enumerate(ep.iterrows()):
            plt.axvspan(row["t_on"], row["t_off"], alpha=0.15, color='orange', lw=0)
            plt.axvline(row["t_on"], ls="--", lw=1.4, color='green', label='on' if i == 0 else None)
            plt.axvline(row["t_off"], ls="--", lw=1.4, color='red',   label='off' if i == 0 else None)
        subtitle = f"Episodes: {len(ep)}"
    else:
        subtitle = "Episodes: 0 (none detected)"

    title_suffix = " (weak adapted)" if is_weak else ""
    plt.title(f"{subject_name} — {trial_id} — {subtitle}{title_suffix}", fontsize=13)
    plt.xlabel("Time (seconds)")
    plt.ylabel("Gate value")
    plt.grid(True, alpha=0.3, linestyle='--')
    plt.legend(loc='upper right', fontsize=9)
    plt.tight_layout()
    plt.show()


def plot_all_trials(subject_name, events_df, logs_df=None, sort=True):
    """
    Plot all trials for a subject.
    Uses logs_df to get the complete trial list (recommended), otherwise falls back to events_df.
    """
    if logs_df is not None and ("trial_id" in logs_df.columns):
        trial_ids = logs_df[logs_df["subject"].astype(str) == str(subject_name)]["trial_id"].astype(str).unique().tolist()
    else:
        trial_ids = events_df[events_df["subject"].astype(str) == str(subject_name)]["trial_id"].astype(str).unique().tolist()

    if sort:
        trial_ids = sorted(trial_ids)

    print(f"Plotting {len(trial_ids)} trials for {subject_name} ...")

    for trial_id in trial_ids:
        plot_trial_with_episodes(subject_name, trial_id, events_df, logs_df=logs_df)


# ─── Main run for one subject ────────────────────────────────
def run_phase4_for_subject(
    subject_name: str,
    method: str = "mad",
    k_on: float = 5.0,
    k_off: float = 3.50,
    pad_s: float = 0.15,
    use_acc_gate: bool = True,
    alpha_acc: float = 1.0,
    compute_sanity_check: bool = True,
    enable_weak_adaptation: bool = True,
    enable_two_stage_refine: bool = True,
):
    imu_path = PHASE2B_DIR / f"{subject_name}__imu_gate.parquet"
    emg_path = PHASE3_NORM_DIR / f"{subject_name}__emg_env_norm.parquet"

    df_imu = pd.DataFrame()
    df_emg = pd.DataFrame()

    if imu_path.exists():
        df_imu = pd.read_parquet(imu_path)
        if "subject" in df_imu.columns:
            df_imu["subject"] = df_imu["subject"].astype(str)
        df_imu["trial_id"] = df_imu["trial_id"].astype(str)

    if emg_path.exists():
        df_emg = pd.read_parquet(emg_path)
        if "subject" in df_emg.columns:
            df_emg["subject"] = df_emg["subject"].astype(str)
        df_emg["trial_id"] = df_emg["trial_id"].astype(str)

    if df_imu.empty and df_emg.empty:
        raise FileNotFoundError(f"No data found for {subject_name}")

    trials_imu = set(df_imu["trial_id"].unique()) if not df_imu.empty else set()
    trials_emg = set(df_emg["trial_id"].unique()) if not df_emg.empty else set()
    all_trials = sorted(trials_imu | trials_emg)

    print(f"Total unique trials: {len(all_trials)}")

    events = []
    logs = []

    is_als_subject = "ALS" in subject_name.upper()

    for trial_id in all_trials:
        use_imu = trial_id in trials_imu

        if use_imu:
            # safe filter (works even if single-subject file)
            if ("subject" in df_imu.columns):
                df_trial = df_imu[(df_imu["subject"] == str(subject_name)) & (df_imu["trial_id"] == str(trial_id))].copy()
            else:
                df_trial = df_imu[(df_imu["trial_id"] == str(trial_id))].copy()
            source_type = "IMU"
        else:
            if ("subject" in df_emg.columns):
                df_trial = df_emg[(df_emg["subject"] == str(subject_name)) & (df_emg["trial_id"] == str(trial_id))].copy()
            else:
                df_trial = df_emg[(df_emg["trial_id"] == str(trial_id))].copy()
            source_type = "EMG_fallback"

        if df_trial.empty:
            continue

        print(f"Processing {trial_id:30} → source: {source_type}")

        task_key = next((k for k in TASK_SETTINGS if k in trial_id.lower()), "default")
        params = TASK_SETTINGS[task_key].copy()

        # Default params
        curr_k_on  = k_on
        curr_k_off = k_off
        curr_peak_mult = 2.5
        curr_min_dur = params["min_duration_s"]
        curr_merge_gap = params["merge_gap_s"]
        curr_sec_merge = 0.8
        curr_alpha = alpha_acc
        smooth_k = 5
        apply_tkeo = False
        is_weak = False

        # Weak/ALS adaptation
        if enable_weak_adaptation:
            if use_imu:
                _, gate_temp, _, _, gate_log_temp = build_gate(
                    df_trial, use_acc_gate=use_acc_gate, alpha=curr_alpha,
                    smooth_kernel=1
                )
            else:
                _, gate_temp, _, _, gate_log_temp = build_gate_from_emg(df_trial, smooth_kernel=1)

            gate_95p = float(gate_log_temp.get("gate_95p", 100.0))

            if is_als_subject or (gate_95p < WEAK_SIGNAL_SETTINGS["gate_95p_threshold"]):
                is_weak = True
                print(f"  → Weak/ALS signal detected (95p={gate_95p:.1f}) → adapting")

                curr_k_on  = WEAK_SIGNAL_SETTINGS["k_on"]
                curr_k_off = WEAK_SIGNAL_SETTINGS["k_off"]
                curr_peak_mult = WEAK_SIGNAL_SETTINGS["peak_multiplier"]
                curr_min_dur *= WEAK_SIGNAL_SETTINGS["min_duration_factor"]
                curr_merge_gap *= WEAK_SIGNAL_SETTINGS["merge_gap_factor"]
                curr_sec_merge = WEAK_SIGNAL_SETTINGS["secondary_merge_gap"]
                smooth_k = WEAK_SIGNAL_SETTINGS["smooth_kernel"]
                apply_tkeo = not use_imu

                if use_imu:
                    curr_alpha = WEAK_SIGNAL_SETTINGS["alpha_acc_boost"]

        # Build final gate
        if use_imu:
            t, gate, gate_source, _, gate_log = build_gate(
                df_trial,
                use_acc_gate=use_acc_gate,
                alpha=curr_alpha,
                smooth_kernel=smooth_k,
                is_weak=is_weak
            )
            gate_log["data_source"] = "imu"
        else:
            t, gate, gate_source, _, gate_log = build_gate_from_emg(
                df_trial,
                smooth_kernel=smooth_k,
                apply_tkeo=apply_tkeo
            )
            gate_log["data_source"] = "emg_only"

        # Gate quality check
        if compute_sanity_check:
            gate_var = float(np.var(gate)) if len(gate) else np.nan
            gate_log["gate_var"] = gate_var
            if np.isfinite(gate_var) and gate_var < 1e-5:
                gate_log["sanity_flag"] = "very_low_variance"

        ep_df, ep_log = detect_episodes_hysteresis(
            t, gate,
            method=method,
            k_on=curr_k_on,
            k_off=curr_k_off,
            min_duration_s=curr_min_dur,
            merge_gap_s=curr_merge_gap,
            pad_s=pad_s,
            peak_multiplier=curr_peak_mult,
            secondary_merge_gap=curr_sec_merge,
        )

        # Post-filtering
        min_dur = WEAK_SIGNAL_SETTINGS["min_valid_duration_als"] if is_weak else 3.5
        min_peak = WEAK_SIGNAL_SETTINGS["min_valid_peak_als"] if is_weak else 50.0

        if not ep_df.empty:
            before = len(ep_df)
            ep_df = ep_df[
                (ep_df["duration_s"] >= min_dur) &
                (ep_df["peak_gate"] >= min_peak)
            ].copy()
            discarded = before - len(ep_df)
            if discarded > 0:
                print(f"  → Discarded {discarded} episodes")
                ep_log["discarded_episodes"] = int(discarded)

        # Optional: Two-stage refine (kept as placeholder behavior)
        if enable_two_stage_refine and len(ep_df) <= 1 and is_weak:
            print("  → Two-stage refine activated (placeholder)")

        log = {
            "subject": subject_name,
            "trial_id": trial_id,
            "task_type": task_key,
            "gate_source": gate_source,
            "data_source": source_type,
            "method": method,
            "is_weak_adapted": bool(is_weak),
            **gate_log,
            **ep_log,
        }

        if ep_df.empty:
            logs.append(log)
            continue

        # recompute peak_gate precisely on final gate
        ep_df["peak_gate"] = [
            float(np.nanmax(gate[(t >= a) & (t <= b)])) if np.any((t >= a) & (t <= b)) else np.nan
            for a, b in zip(ep_df["t_on"], ep_df["t_off"])
        ]

        ep_df = ep_df.sort_values("t_on").reset_index(drop=True)
        ep_df["episode_id"] = range(1, len(ep_df) + 1)

        for _, row in ep_df.iterrows():
            events.append({
                "subject": subject_name,
                "trial_id": trial_id,
                "episode_id": int(row["episode_id"]),
                "t_on": float(row["t_on"]),
                "t_off": float(row["t_off"]),
                "duration_s": float(row["duration_s"]),
                "peak_gate": float(row["peak_gate"]) if pd.notna(row["peak_gate"]) else np.nan,
                "thr_on": float(row["thr_on"]),
                "thr_off": float(row["thr_off"]),
                "gate_source": gate_source,
                "task_type": task_key,
                "data_source": source_type,
                "is_weak": bool(is_weak),
            })

        logs.append(log)

    events_df = pd.DataFrame(events)
    logs_df   = pd.DataFrame(logs)

    events_df.to_parquet(PHASE4_DIR / f"{subject_name}__episodes.parquet", index=False)
    logs_df.to_csv(PHASE4_DIR / f"{subject_name}__episodes_log.csv", index=False)

    print(f"\nProcessing {subject_name} completed.")
    return events_df, logs_df


# ─── Example usage ───────────────────────────────────────────────
events_df, logs_df = run_phase4_for_subject("ALS_Subject_4", enable_weak_adaptation=True)

# ✅ NEW: Plot every trial (even if no episodes were detected)
plot_all_trials("ALS_Subject_4", events_df, logs_df)
